<a href="https://colab.research.google.com/github/anamitra-tech/ML-Projects/blob/main/AegisDrone%20%E2%80%94%20AI-based%20Drone%20Threat%20Detection%20%26%20Classification%20System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/f4c2b4n755-1.zip" -d "/content/drive/MyDrive/DroneRF"

In [ ]:
!pip install rarfile

In [ ]:
!find /content/drive/MyDrive -name "*.rar"

/content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_L1.rar
/content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_H1.rar
/content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_L2.rar
/content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/FR Data_00000_H2.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10000_L.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10000_H.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10011_L.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10011_H.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10010_L.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10001_L.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10001_H.rar
/content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10010_H.rar
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10110_L.rar
/content/drive/MyDrive/Drone

In [ ]:
import os
from pathlib import Path

base = Path("/content/drive/MyDrive/DroneRF/DroneRF")

for rar_file in base.rglob("*.rar"):
    print(f"Extracting: {rar_file}")
    os.system(f'unrar x -o+ "{rar_file}" "{rar_file.parent}/"')

Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_L1.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_H1.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_L2.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/FR Data_00000_H2.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10000_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10000_H.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10011_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10011_H.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10010_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10001_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10001_H.rar
Extracting: /content/drive/MyDrive/

In [11]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║        REAL-TIME AI ANTI-DRONE CLASSIFICATION & THREAT DETECTION            ║
# ║                         PRODUCTION SYSTEM  v9                               ║
# ║                                                                              ║
# ║  ROOT CAUSES FIXED vs v8 (all caused 100% OPEN_SET_UNKNOWN):               ║
# ║                                                                              ║
# ║  BUG-1  EVM distance mismatch  [CRITICAL]                                  ║
# ║    v8:  Weibull fit used COSINE distances between class points              ║
# ║         but inference used L2(x, class_mean) — 9.8× scale mismatch.       ║
# ║         Result: CDF(L2_dist) ≈ 1.0 always → P(include) ≈ 0 → all rejected.║
# ║    FIX:  Use same metric everywhere (L2). Fit on tail of L2 distances       ║
# ║         from class centroid. Calibrate threshold on held-out val set.      ║
# ║                                                                              ║
# ║  BUG-2  EVM Weibull shape collapse  [CRITICAL]                             ║
# ║    v8:  Weibull shape ≈ 66–167 (nearly a spike). 5% tail of 160 points    ║
# ║         = 8 samples in a very tight range → degenerate fit.               ║
# ║    FIX:  tail_size=0.30 (30% of class, well-sampled); floc fit freed;     ║
# ║         fallback to percentile gate when Weibull fit fails.               ║
# ║                                                                              ║
# ║  BUG-3  Hard-rejection stacking  [ARCHITECTURAL]                           ║
# ║    v8:  4 independent binary gates in series (EVM AND vacuity AND conf     ║
# ║         AND anomaly). P(pass) = 0.8^4 = 0.41 even at 80% pass rate each. ║
# ║    FIX:  Single soft scoring function. No hard gates except final          ║
# ║         OPEN_SET threshold tuned on validation set.                        ║
# ║                                                                              ║
# ║  ARCHITECTURE DECISIONS:                                                    ║
# ║    • Soft fusion score = weighted blend of classifier confidence,          ║
# ║      EVM inclusion, and anomaly normality                                  ║
# ║    • EVM as a SCORE (0–1), not a gate                                      ║
# ║    • EDL vacuity as a PENALTY, not a gate                                  ║
# ║    • Single validation-calibrated threshold for OPEN_SET decision          ║
# ║    • Temperature scaling for classifier calibration                        ║
# ║    • All thresholds derived from held-out validation set                   ║
# ║    • Full pipeline trace logging for every decision                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 0  ·  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR   = "/content/drive/MyDrive/DroneRF/DroneRF"
OUTPUT_CSV = "dronerf_features_v9.csv"
DB_PATH    = "antidrone_db_v9.json"
LOG_PATH   = "antidrone_audit_v9.jsonl"

RANDOM_SEED           = 42
WINDOW_SIZE           = 8192
STEP_SIZE             = 4096
FS                    = 10e6
TARGET_TOTAL          = 4500
MAX_SEGMENTS_PER_FILE = 50

# ── Fusion weights (must sum to 1.0) ─────────────────────────────────────────
# These govern how soft_score = w_clf*clf_conf + w_evm*evm_score + w_norm*normality
FUSION_W_CLF      = 0.55   # classifier confidence (RF×GBT×GBP geometric mean)
FUSION_W_EVM      = 0.25   # EVM inclusion probability
FUSION_W_NORMALITY= 0.20   # 1 - normalised_anomaly_score
assert abs(FUSION_W_CLF + FUSION_W_EVM + FUSION_W_NORMALITY - 1.0) < 1e-9

# ── Decision thresholds (all calibrated on validation set) ───────────────────
# OPEN_SET_THRESHOLD: soft_score below this → OPEN_SET_UNKNOWN
# This replaces ALL hard binary gates from v8.
OPEN_SET_THRESHOLD    = 0.40   # initial value; overridden by calibrate_thresholds()
FRIENDLY_THRESHOLD    = 0.65   # soft_score above this → route to FRIENDLY / BG fast-path
THREAT_SCORE_THRESH   = 0.72   # anomaly_score above this → threat path
HIGH_THREAT_THRESHOLD = 0.80   # mean_threat for CONFIRMED status
CONFIRMED_THREAT_OBS  = 5
AUTO_CLASSIFY_CONF    = 0.35
SIMILARITY_THRESHOLD  = 0.88

# ── EVM (fixed from v8) ───────────────────────────────────────────────────────
EVM_TAIL_SIZE         = 0.30   # 30% of class (was 5% → degenerate)
# EVM_COVER_THRESHOLD removed — EVM now outputs a soft score, no hard gate

# ── Fingerprint / trust ───────────────────────────────────────────────────────
HASH_N_BINS           = 200
HASH_CLIP             = 50.0
HASH_TOP_FEATURES     = 12
TRUST_MIN_OBSERVATIONS= 8
TRUST_MAX_VARIANCE    = 0.60

# ── GBP temperature ───────────────────────────────────────────────────────────
GBP_TEMPERATURE       = 0.5

# ── Laplace ───────────────────────────────────────────────────────────────────
LAPLACE_PRIOR_PRECISION = 1.0
LAPLACE_N_SAMPLES       = 256
ONLINE_ETA              = 0.05

# ── VBGMM ────────────────────────────────────────────────────────────────────
VBGMM_MAX_COMPONENTS    = 6
VBGMM_MAX_ITER          = 400

# ── EvidentialNet ────────────────────────────────────────────────────────────
EDL_HIDDEN        = (256, 128, 64)
EDL_DROPOUT       = 0.10
EDL_LR            = 1e-3
EDL_WEIGHT_DECAY  = 1e-4
EDL_BATCH_SIZE    = 128
EDL_MAX_EPOCHS    = 200
EDL_ES_PATIENCE   = 15
EDL_ANNEAL_START  = 10
EDL_KL_WEIGHT     = 0.001


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1  ·  IMPORTS
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys

def _pip(*pkgs):
    for flags in [[], ["--break-system-packages"]]:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", *pkgs, "-q", *flags],
            capture_output=True
        )
        if r.returncode == 0:
            return

_pip("numpy", "pandas", "scipy", "scikit-learn", "imbalanced-learn",
     "matplotlib", "seaborn", "tqdm", "torch")

import gc, os, re, time, warnings, hashlib, json, copy, logging
from collections     import defaultdict, deque, Counter
from dataclasses     import dataclass, field
from pathlib         import Path
from typing          import Dict, List, Optional, Tuple, Any

import numpy             as np
import pandas            as pd
import matplotlib;       matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn           as sns
from matplotlib.patches  import Patch

import torch
import torch.nn          as nn
import torch.optim       as optim
import torch.nn.functional as F
from torch.utils.data    import DataLoader, TensorDataset

from scipy.stats   import kurtosis, skew, weibull_min
from scipy.signal  import hilbert, welch, stft
from scipy.linalg  import cho_factor, cho_solve
from scipy.special import digamma as sp_digamma
from scipy.spatial.distance import cdist

from sklearn.calibration       import CalibratedClassifierCV
from sklearn.decomposition     import PCA
from sklearn.ensemble          import (RandomForestClassifier,
                                        GradientBoostingClassifier,
                                        IsolationForest)
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model      import LogisticRegression, SGDClassifier
from sklearn.metrics           import (accuracy_score, f1_score,
                                        classification_report,
                                        confusion_matrix, roc_auc_score)
from sklearn.mixture           import BayesianGaussianMixture
from sklearn.model_selection   import (train_test_split, StratifiedKFold,
                                        cross_val_score)
from sklearn.preprocessing     import RobustScaler, label_binarize
from sklearn.svm               import SVC
from imblearn.over_sampling    import SMOTE
from tqdm                      import tqdm

warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Audit logger
_audit = logging.getLogger("antidrone.v9")
_audit.setLevel(logging.DEBUG)
_fh = logging.FileHandler(LOG_PATH, mode="w")
_fh.setFormatter(logging.Formatter("%(message)s"))
_audit.addHandler(_fh)

def audit(event: str, **kw):
    _audit.debug(json.dumps({"ts": round(time.time(), 4), "event": event, **kw}))

print(f"✓ Imports ready  |  device={DEVICE}  |  Python {sys.version.split()[0]}")

# ── Class taxonomy ────────────────────────────────────────────────────────────
CLASS_NAMES = {0: "Background RF", 1: "AR Drone", 2: "Phantom Drone"}
N_CLASSES   = len(CLASS_NAMES)
BG_NAME     = CLASS_NAMES[0]
FOLDER_MAP  = {"background": 0, "ar drone": 1, "ar_drone": 1,
               "ardrone": 1, "phantom": 2}
BUI_MAP     = {"00000": 0, "10000": 1, "10001": 1, "10010": 1,
               "10011": 1, "10100": 1, "10101": 1, "10110": 1,
               "11000": 2, "11001": 2, "11010": 2}

DECISION_ICONS = {
    "FRIENDLY_DRONE":    "🟢",
    "BACKGROUND":        "⚪",
    "POTENTIAL_THREAT":  "🔴",
    "CONFIRMED_THREAT":  "🚨",
    "SAFE_NEW_DRONE":    "🔵",
    "TRUSTED_NEW_DRONE": "🔷",
    "UNKNOWN_MONITOR":   "🟡",
    "AUTO_AR_DRONE":     "🟩",
    "AUTO_PHANTOM_DRONE":"🟦",
    "OPEN_SET_UNKNOWN":  "❓",
}


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2  ·  FEATURE ENGINEERING  (RF=52 + Flight=18 + Comm=12 → 82)
# ─────────────────────────────────────────────────────────────────────────────

RF_FEATURE_NAMES = [
    "amp_mean","amp_std","amp_var","amp_min","amp_max",
    "amp_range","amp_kurtosis","amp_skew",
    "signal_power_db","IQ_corr","I_power","Q_power","iq_power_ratio","iq_corr_sq",
    "peak_freq_hz","bandwidth_hz","spectral_entropy",
    "spectral_centroid","spectral_spread","spectral_rolloff_85",
    "psd_mean_db","psd_max_db",
    "ifreq_mean","ifreq_std","ifreq_range","ifreq_kurtosis",
    "energy_band1","energy_band2","energy_band3","energy_band4",
    "stft_flux_var","stft_sub1_var","stft_sub2_var","stft_sub3_var","stft_sub4_var",
    "spec_kurtosis","spec_skewness","l_kurtosis","spec_flatness","stft_entropy",
    "am_depth","crest_factor","phase_jitter","spec_asymmetry",
    "acf_short","acf_medium","acf_long","acf_ratio",
    "kurt_entropy_product","snr_like_db","spectral_variance","temporal_kurtosis",
]
N_RF = len(RF_FEATURE_NAMES)
assert N_RF == 52

FLIGHT_FEATURE_NAMES = [
    "speed_mean","speed_std","speed_max",
    "accel_mean","accel_std","accel_max",
    "altitude_mean","altitude_std",
    "heading_change_rate","heading_std",
    "path_curvature","loiter_fraction",
    "approach_vector_sin","approach_vector_cos",
    "proximity_score","hover_time_fraction",
    "trajectory_entropy","maneuver_intensity",
]
N_FLIGHT = len(FLIGHT_FEATURE_NAMES)
assert N_FLIGHT == 18

COMM_FEATURE_NAMES = [
    "tx_rate_hz","tx_burst_ratio","protocol_entropy",
    "command_interval_mean","command_interval_std","telemetry_rate_hz",
    "encryption_flag","freq_hop_count","channel_dwell_mean",
    "control_link_snr","video_link_active","swarm_signal_flag",
]
N_COMM = len(COMM_FEATURE_NAMES)
assert N_COMM == 12

ALL_FEATURE_NAMES = RF_FEATURE_NAMES + FLIGHT_FEATURE_NAMES + COMM_FEATURE_NAMES
N_FEATURES        = len(ALL_FEATURE_NAMES)
FEAT_IDX          = {n: i for i, n in enumerate(ALL_FEATURE_NAMES)}
print(f"✓ Features: RF={N_RF} + flight={N_FLIGHT} + comm={N_COMM} = {N_FEATURES}")


def _pearson(x, y):
    xm = x - x.mean(); ym = y - y.mean()
    return float(np.dot(xm, ym) / ((np.dot(xm, xm) * np.dot(ym, ym)) ** 0.5 + 1e-12))

def _pct(arr, q):
    c = arr[np.isfinite(arr)]
    return float(np.percentile(c, q)) if len(c) > 0 else 0.0


def extract_rf_features(real_seg: np.ndarray, fs: float = FS) -> np.ndarray:
    real     = real_seg.astype(np.float64)
    N        = len(real)
    analytic = hilbert(real)
    I, Q     = analytic.real, analytic.imag
    envelope = np.abs(analytic)
    out      = np.empty(N_RF, dtype=np.float32)

    amp_mean = float(envelope.mean());  amp_std = float(envelope.std())
    amp_min  = float(envelope.min());   amp_max = float(envelope.max())
    amp_ptp  = amp_max - amp_min
    amp_kurt = float(kurtosis(envelope)) if amp_std > 1e-8 else 0.0
    amp_skew = float(skew(envelope))     if amp_std > 1e-8 else 0.0
    out[0:8] = [amp_mean, amp_std, amp_std**2, amp_min, amp_max,
                amp_ptp, amp_kurt, amp_skew]

    I_pow   = float(np.dot(I, I) / N);  Q_pow = float(np.dot(Q, Q) / N)
    rms     = float((np.dot(envelope, envelope) / N) ** 0.5)
    pow_db  = float(10.0 * np.log10(np.dot(envelope, envelope) / N + 1e-12))
    iq_corr = _pearson(I, Q) if amp_std > 1e-12 else 0.0
    out[8:14] = [pow_db, iq_corr, I_pow, Q_pow, I_pow / (Q_pow + 1e-12), iq_corr**2]

    nperseg = min(512, N // 4)
    fw, psd = welch(envelope, fs=fs, nperseg=nperseg, noverlap=nperseg//2, return_onesided=True)
    pa      = np.clip(np.abs(psd), 1e-12, None);  pa_sum = pa.sum()
    pd_db   = 10.0 * np.log10(pa);  pk = int(pa.argmax())
    above   = fw[pd_db > pd_db[pk] - 10.0]
    bw      = float(above.max() - above.min()) if len(above) > 1 else 0.0
    pn      = pa / pa_sum
    entropy = float(-np.dot(pn, np.log2(pn + 1e-12)))
    cen     = float(np.dot(fw, pa) / pa_sum)
    spread  = float(np.sqrt(np.dot((fw - cen) ** 2, pa) / pa_sum))
    cs      = np.cumsum(pa)
    rol     = min(int(np.searchsorted(cs, 0.85 * cs[-1])), len(fw) - 1)
    out[14:22] = [fw[pk], bw, entropy, cen, spread, fw[rol],
                  float(pd_db.mean()), float(pd_db.max())]

    ifreq = np.diff(np.unwrap(np.angle(analytic)))
    if len(ifreq) >= 2 and ifreq.std() > 1e-8:
        out[22:26] = [float(ifreq.mean()), float(ifreq.std()),
                      float(ifreq.max() - ifreq.min()), float(kurtosis(ifreq))]
    else:
        out[22:26] = [0., 0., 0., 0.]

    q_sz = max(1, len(pa) // 4)
    out[26:30] = [pa[:q_sz].sum()/pa_sum, pa[q_sz:2*q_sz].sum()/pa_sum,
                  pa[2*q_sz:3*q_sz].sum()/pa_sum, pa[3*q_sz:].sum()/pa_sum]

    stft_np = min(128, N // 4)
    _, _, Zxx = stft(envelope, fs=fs, nperseg=stft_np, noverlap=stft_np//2, return_onesided=True)
    Sxx      = np.abs(Zxx)**2 + 1e-12;  fm = Sxx.mean(0)
    out[30]  = float(np.diff(fm).var())
    bsz      = max(1, Sxx.shape[0] // 4)
    for b in range(4): out[31+b] = float(Sxx[b*bsz:(b+1)*bsz,:].mean(0).var())

    pa_s      = np.sort(pa)
    spec_kurt = float(kurtosis(pa));  spec_skew = float(skew(pa))
    L2 = pa_s[1::2].mean() - pa_s[::2].mean()
    L4 = (pa_s[3::4].mean() - 3*pa_s[2::4].mean()
          + 3*pa_s[1::4].mean() - pa_s[::4].mean())
    l_kurt    = float(L4 / (L2 + 1e-12))
    spec_flat = float(np.exp(np.log(pa + 1e-12).mean() - np.log(pa.mean() + 1e-12)))
    Sxx_n     = Sxx.mean(1);  Sxx_n /= Sxx_n.sum() + 1e-12
    stft_ent  = float(-np.dot(Sxx_n, np.log2(Sxx_n + 1e-12)))
    out[35:40] = [spec_kurt, spec_skew, l_kurt, spec_flat, stft_ent]

    out[40:44] = [
        float((envelope.max() - envelope.min()) / (amp_mean + 1e-12)),
        float(envelope.max() / (rms + 1e-12)),
        float(np.diff(ifreq).std()) if len(ifreq) >= 2 else 0.0,
        float((pa[fw >= cen].sum() - pa[fw < cen].sum()) / (pa_sum + 1e-12)),
    ]

    en = ((envelope - amp_mean) / (amp_std + 1e-9))[:min(512, N)]
    N_s = len(en)
    ls, lm, ll = min(50, N_s//10), min(200, N_s//3), min(400, N_s//2)
    acf_s = _pearson(en[:N_s-ls], en[ls:])
    acf_m = _pearson(en[:N_s-lm], en[lm:])
    acf_l = _pearson(en[:N_s-ll], en[ll:])
    out[44:48] = [acf_s, acf_m, acf_l, acf_m / (acf_s + 1e-6)]

    top10 = pa_s[::-1][:max(1, len(pa_s)//10)].mean()
    bot50 = pa_s[::-1][len(pa_s)//2:].mean()
    out[48:52] = [amp_kurt * entropy,
                  float(10.0 * np.log10(top10 / (bot50 + 1e-12))),
                  float(pa.var()),
                  float(kurtosis(real))]

    return np.nan_to_num(out, nan=0., posinf=0., neginf=0.)


def safe_extract_rf(seg: np.ndarray, fs: float = FS) -> np.ndarray:
    try:
        return extract_rf_features(seg, fs)
    except Exception as e:
        safe_extract_rf._n = getattr(safe_extract_rf, "_n", 0) + 1
        if safe_extract_rf._n <= 3:
            print(f"  [safe_extract #{safe_extract_rf._n}]: {e}")
        return np.zeros(N_RF, dtype=np.float32)


def fuse_features(rf: np.ndarray,
                  flight: Optional[np.ndarray] = None,
                  comm:   Optional[np.ndarray] = None) -> np.ndarray:
    fl = flight.astype(np.float32) if flight is not None else np.zeros(N_FLIGHT, dtype=np.float32)
    co = comm.astype(np.float32)   if comm   is not None else np.zeros(N_COMM,   dtype=np.float32)
    return np.concatenate([rf.astype(np.float32), fl, co])


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3  ·  DATA INGESTION
# ─────────────────────────────────────────────────────────────────────────────

def folder_to_class(name):
    n = name.lower().strip()
    for k, v in FOLDER_MAP.items():
        if k in n: return v
    return None

def bui_to_class(fname):
    m = re.search(r"\d{5}", Path(fname).stem)
    return BUI_MAP.get(m.group(0)) if m else None


def generate_synthetic_dataset(n_per_class: int = 1500,
                                rng_seed: int = RANDOM_SEED) -> pd.DataFrame:
    """
    Realistic synthetic DroneRF data.
    Class-discriminative features have strong separation;
    remaining features are independent Gaussian noise.
    This mirrors the real DroneRF structure where 5 features
    carry >90% of class information (as seen in MI ranking).
    """
    rng = np.random.default_rng(rng_seed)
    print(f"\n  ⚙️  Synthetic dataset  (n_per_class={n_per_class})")

    # Discriminative feature profiles — matched to real DroneRF statistics
    profiles = {
        0: dict(signal_power_db=(-30, 6), spectral_entropy=(3.5, 0.6),
                bandwidth_hz=(0.4e6, 0.15e6), ifreq_std=(0.05, 0.02),
                amp_kurtosis=(0.3, 0.4)),
        1: dict(signal_power_db=(-18, 4), spectral_entropy=(5.5, 0.5),
                bandwidth_hz=(2.0e6, 0.4e6), ifreq_std=(0.8, 0.15),
                amp_kurtosis=(2.5, 0.8)),
        2: dict(signal_power_db=(-12, 4), spectral_entropy=(6.8, 0.4),
                bandwidth_hz=(4.2e6, 0.6e6), ifreq_std=(1.6, 0.20),
                amp_kurtosis=(4.2, 0.9)),
    }
    rows, labels = [], []
    for cls, prof in profiles.items():
        for _ in range(n_per_class):
            fv = rng.standard_normal(N_FEATURES).astype(np.float32) * 0.1
            for feat, (mean, std) in prof.items():
                if feat in FEAT_IDX:
                    fv[FEAT_IDX[feat]] = float(rng.normal(mean, std))
            for fn in ["bandwidth_hz", "ifreq_std", "spectral_entropy",
                       "amp_std", "amp_var"]:
                if fn in FEAT_IDX:
                    fv[FEAT_IDX[fn]] = abs(fv[FEAT_IDX[fn]])
            rows.append(fv); labels.append(cls)

    X   = np.array(rows, dtype=np.float32)
    df  = pd.DataFrame(X, columns=ALL_FEATURE_NAMES)
    df.insert(0, "label_int",  labels)
    df.insert(1, "label_name", [CLASS_NAMES[c] for c in labels])
    df.insert(2, "source_file", ["synthetic"] * len(labels))
    df  = df.sample(frac=1, random_state=rng_seed).reset_index(drop=True)
    print(f"  ✓ {len(df):,} rows  ({n_per_class} × {N_CLASSES} classes)")
    return df


def build_or_load_dataset(data_dir: str, output_csv: str = OUTPUT_CSV) -> pd.DataFrame:
    cache = Path(output_csv)
    if cache.exists():
        try:
            df = pd.read_csv(output_csv)
            if len([c for c in df.columns if c in RF_FEATURE_NAMES]) == N_RF \
                    and df["amp_std"].var() > 1e-4:
                for col in ALL_FEATURE_NAMES:
                    if col not in df.columns: df[col] = 0.0
                print(f"⚡ Loaded cache: {output_csv}  ({len(df):,} rows)")
                return df
        except Exception:
            cache.unlink(missing_ok=True)

    if data_dir and Path(data_dir).exists():
        print(f"\n{'='*60}\nBUILDING DATASET FROM {data_dir}\n{'='*60}")
        try:
            cf = {}
            root = Path(data_dir)
            for subdir in sorted(root.iterdir()):
                if not subdir.is_dir(): continue
                c = folder_to_class(subdir.name)
                if c is None: continue
                files = sorted(subdir.rglob("*.csv"))
                if files: cf[c] = files
            if not cf:
                for fp in sorted(root.rglob("*.csv")):
                    c = bui_to_class(fp.name)
                    if c is not None: cf.setdefault(c, []).append(fp)
            if not cf: raise RuntimeError("No CSV files found")

            q_base = TARGET_TOTAL // len(cf)
            rng    = np.random.default_rng(RANDOM_SEED)
            rows, labels, files_used = [], [], []
            for cls_int, flist in sorted(cf.items()):
                shuffled = list(flist); rng.shuffle(shuffled)
                count = 0
                for fp in shuffled:
                    if count >= q_base: break
                    try:
                        raw = pd.read_csv(fp, header=None, dtype=np.float32).values.ravel()
                    except Exception: continue
                    start = WINDOW_SIZE
                    while start + WINDOW_SIZE <= len(raw) and count < q_base:
                        rf_fv  = safe_extract_rf(raw[start:start+WINDOW_SIZE])
                        fv_all = fuse_features(rf_fv)
                        rows.append(fv_all); labels.append(cls_int)
                        files_used.append(fp.name)
                        start += STEP_SIZE; count += 1
            X  = np.array(rows, dtype=np.float32)
            df = pd.DataFrame(X, columns=ALL_FEATURE_NAMES)
            df.insert(0, "label_int",  labels)
            df.insert(1, "label_name", [CLASS_NAMES[c] for c in labels])
            df.insert(2, "source_file", files_used)
            df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
            df.to_csv(output_csv, index=False)
            print(f"✓ Saved {len(df):,} rows → {output_csv}")
            return df
        except Exception as e:
            print(f"  [WARN] Real data failed: {e}. Using synthetic.")

    df = generate_synthetic_dataset()
    df.to_csv(output_csv, index=False)
    return df


def prepare_data(df: pd.DataFrame):
    X_all = np.nan_to_num(
        df[ALL_FEATURE_NAMES].fillna(0).values.astype(np.float32),
        nan=0., posinf=0., neginf=0.
    )
    y_all  = df["label_int"].values.astype(np.int64)
    counts = {c: int((y_all == c).sum()) for c in np.unique(y_all)}
    valid  = [c for c, n in counts.items() if n >= 6]
    mask   = np.isin(y_all, valid)
    X_use, y_use = X_all[mask], y_all[mask]
    lmap   = {old: new for new, old in enumerate(sorted(valid))}
    y_map  = np.array([lmap[yi] for yi in y_use], dtype=np.int64)
    classes = [CLASS_NAMES[c] for c in sorted(valid)]
    print(f"\n  Classes: {len(classes)}")
    for i, cn in enumerate(classes):
        print(f"    [{i}] {cn}  ({(y_map==i).sum()} samples)")
    return X_use, y_map, lmap, classes, len(classes)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4  ·  FEATURE SELECTION
# ─────────────────────────────────────────────────────────────────────────────

def validate_and_select_features(X: np.ndarray, y: np.ndarray,
                                  top_k: Optional[int] = None):
    print(f"\n{'='*60}\nFEATURE SELECTION\n{'='*60}")
    sc  = RobustScaler()
    X_s = np.nan_to_num(sc.fit_transform(X), nan=0., posinf=0., neginf=0.)

    var_mask = X_s.var(0) > 1e-15
    print(f"  Zero-variance dropped: {(~var_mask).sum()}")

    if var_mask.sum() >= 3:
        pca = PCA(n_components=min(10, var_mask.sum()))
        pca.fit(X_s[:, var_mask])
        cum = np.cumsum(pca.explained_variance_ratio_)
        for k in [3, 5, 10]:
            k2  = min(k, len(cum))
            tag = "✓" if cum[k2-1] > 0.60 else "△" if cum[k2-1] > 0.40 else "✗"
            print(f"    Top-{k2:>2} PCs: {cum[k2-1]:.3f}  [{tag}]")

    mi      = mutual_info_classif(X_s, y, random_state=RANDOM_SEED)
    top_idx = np.argsort(mi)[::-1]
    print(f"\n  Top-15 features by MI:")
    for rank, i in enumerate(top_idx[:15], 1):
        s = "★★" if mi[i]>0.30 else "★" if mi[i]>0.10 else "○" if mi[i]>0.05 else "△"
        print(f"    {rank:>2}. {ALL_FEATURE_NAMES[i]:<35}  {mi[i]:.4f}  {s}")

    selected_idx = top_idx if top_k is None else top_idx[:top_k]
    scaler_sel   = RobustScaler()
    X_sel        = np.nan_to_num(scaler_sel.fit_transform(X[:, selected_idx]),
                                  nan=0., posinf=0., neginf=0.)
    return X_sel, selected_idx, scaler_sel, mi


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5  ·  PROBABILISTIC MODELS
# ─────────────────────────────────────────────────────────────────────────────

# ── 5a. Gaussian Bayes Posterior ──────────────────────────────────────────────

class GaussianBayesPosterior:
    """
    Class-conditional diagonal Gaussian.
    p(class|x) ∝ p(x|class) × π_class.
    Temperature τ<1 sharpens known-class predictions.
    Points far from all class means → all likelihoods low → naturally novel.
    """
    def __init__(self, temperature: float = GBP_TEMPERATURE,
                 var_smoothing_frac: float = 1e-3):
        self.tau = temperature;  self.vsf = var_smoothing_frac
        self.fitted = False

    def fit(self, X: np.ndarray, y: np.ndarray) -> "GaussianBayesPosterior":
        t0 = time.time(); classes = np.unique(y); self.classes_ = classes
        n_total = len(y); smooth = self.vsf * X.var(0).mean()
        self.mu_: Dict[int, np.ndarray] = {}
        self.var_: Dict[int, np.ndarray] = {}
        self.log_prior_: Dict[int, float] = {}
        for k in classes:
            Xk = X[y == k]
            self.mu_[k]  = Xk.mean(0)
            self.var_[k] = Xk.var(0) + smooth
            self.log_prior_[k] = float(np.log(len(Xk) / n_total))
        self.fitted = True
        print(f"  ✓ GBP  ({time.time()-t0:.2f}s)  τ={self.tau}  classes={list(classes)}")
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=np.float64)
        log_posts = np.stack([
            -0.5 * ((X - self.mu_[k])**2 / self.var_[k]).sum(1) / self.tau
            - 0.5 * np.log(2 * np.pi * self.var_[k]).sum() / self.tau
            + self.log_prior_[k]
            for k in self.classes_
        ], axis=1)
        log_posts -= log_posts.max(1, keepdims=True)
        probs = np.exp(log_posts);  probs /= probs.sum(1, keepdims=True)
        return probs

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.predict_proba(X).argmax(1)


# ── 5b. Temperature Scaling Calibrator ───────────────────────────────────────

class TemperatureScaler:
    """
    Post-hoc calibration via a single learned temperature parameter T.
    Calibrated logits = logits / T.
    Trained by minimizing NLL on the validation set.
    Fixes overconfident softmax predictions (common when RF ≈ 97% train acc
    but produces 0.99+ confidence on misclassified samples).
    """
    def __init__(self):
        self.T = 1.0

    def fit(self, logits: np.ndarray, y: np.ndarray,
            n_iter: int = 200, lr: float = 0.01) -> "TemperatureScaler":
        T = torch.nn.Parameter(torch.ones(1))
        opt = torch.optim.LBFGS([T], lr=lr, max_iter=n_iter)
        log_t = torch.from_numpy(logits.astype(np.float32))
        y_t   = torch.from_numpy(y.astype(np.int64))

        def closure():
            opt.zero_grad()
            loss = F.cross_entropy(log_t / T.clamp(min=0.1), y_t)
            loss.backward()
            return loss

        opt.step(closure)
        self.T = float(T.item())
        print(f"  ✓ TemperatureScaler  T={self.T:.4f}")
        return self

    def calibrate(self, logits: np.ndarray) -> np.ndarray:
        scaled = logits / self.T
        e = np.exp(scaled - scaled.max(1, keepdims=True))
        return e / e.sum(1, keepdims=True)


# ── 5c. Laplace Approximation ─────────────────────────────────────────────────

class LaplaceApproximation:
    def __init__(self, prior_precision: float = LAPLACE_PRIOR_PRECISION,
                 n_samples: int = LAPLACE_N_SAMPLES):
        self.alpha = prior_precision;  self.n_samples = n_samples
        self.fitted = False

    def fit(self, lr_model, X_train: np.ndarray,
            y_train: np.ndarray, n_classes: int) -> "LaplaceApproximation":
        t0 = time.time();  self.n_classes = n_classes
        D = X_train.shape[1]
        self.W_map = lr_model.coef_.astype(np.float64)
        self.b_map = lr_model.intercept_.astype(np.float64)
        Z = X_train @ self.W_map.T + self.b_map
        Z -= Z.max(1, keepdims=True)
        eZ = np.exp(Z);  probs = eZ / eZ.sum(1, keepdims=True)
        self.chol_factors: List = []
        for k in range(n_classes):
            pi_k = probs[:, k].clip(1e-7, 1 - 1e-7)
            wk   = pi_k * (1.0 - pi_k)
            H_k  = (X_train * wk[:, None]).T @ X_train + self.alpha * np.eye(D)
            try:
                cf = cho_factor(H_k, lower=False, check_finite=False)
                self.chol_factors.append(("chol", cf, H_k))
            except Exception:
                self.chol_factors.append(("pinv", np.linalg.pinv(H_k), H_k))
        self.fitted = True
        print(f"  ✓ Laplace  ({time.time()-t0:.2f}s)  D={D}  classes={n_classes}")
        return self

    def predictive_variance(self, X: np.ndarray) -> float:
        if not self.fitted: return 0.0
        X = np.asarray(X, dtype=np.float64);  C = self.n_classes
        samples = np.zeros((self.n_samples, X.shape[0], C))
        for k in range(C):
            kind, factor, H = self.chol_factors[k]
            D = self.W_map.shape[1];  z = np.random.randn(self.n_samples, D)
            if kind == "chol":
                try:    v = cho_solve(factor, z.T, check_finite=False).T
                except: v = z * (1.0 / (np.diag(H) + 1e-8))
            else:
                try:    L = np.linalg.cholesky(factor + 1e-8 * np.eye(D)); v = (L @ z.T).T
                except: v = z * np.sqrt(np.diag(factor) + 1e-8)
            samples[:, :, k] = (X @ (self.W_map[k] + v).T + self.b_map[k]).T
        Z = samples - samples.max(-1, keepdims=True)
        p = np.exp(Z);  p /= p.sum(-1, keepdims=True)
        return float(p.var(0).mean())

    def online_update(self, x_new: np.ndarray, y_new: int, eta: float = ONLINE_ETA):
        if not self.fitted: return
        x = np.asarray(x_new, dtype=np.float64).ravel()
        kind, factor, H = self.chol_factors[y_new]
        H_new = H + eta * np.outer(x, x)
        try:
            cf_new = cho_factor(H_new, lower=False, check_finite=False)
            self.chol_factors[y_new] = ("chol", cf_new, H_new)
        except Exception: pass


# ── 5d. Extreme Value Machine (FIXED) ────────────────────────────────────────

class ExtremValueMachine:
    """
    FIXED vs v8:
      BUG-1 FIX: Use L2 distance consistently for both fit and inference.
                 v8 used cosine for fit and L2 for inference → 9.8× scale mismatch.
      BUG-2 FIX: tail_size=0.30 (was 0.05 → ~8 samples → degenerate Weibull shape≈160).
                 With shape≈160 the Weibull is nearly a spike; any point outside
                 the tiny tail range gets CDF≈1 → P(include)≈0.

    Now returns a SOFT SCORE in [0, 1] — NOT a binary gate.
    The score is the mean inclusion probability across all class models.
    This feeds into the soft fusion score as one component.

    Threshold is calibrated on validation set, not hard-coded.
    """
    def __init__(self, tail_size: float = EVM_TAIL_SIZE):
        self.tail_size = tail_size
        self.weibull_params: Dict[int, Tuple] = {}
        self.class_means:    Dict[int, np.ndarray] = {}
        self.class_stds:     Dict[int, float] = {}
        self.fitted = False

    def fit(self, X: np.ndarray, y: np.ndarray) -> "ExtremValueMachine":
        t0 = time.time();  classes = np.unique(y)
        for k in classes:
            Xk  = X[y == k]
            mu  = Xk.mean(0)
            self.class_means[k] = mu
            # L2 distances from class centroid (same metric used at inference)
            dists = np.linalg.norm(Xk - mu, axis=1)
            self.class_stds[k] = float(dists.std()) + 1e-6
            # Tail = largest distances (most extreme within-class points)
            n_tail = max(3, int(self.tail_size * len(dists)))
            tail   = np.sort(dists)[-n_tail:]
            try:
                # Free loc parameter for better fit
                shape, loc, scale = weibull_min.fit(tail)
                # Sanity check: degenerate fit
                if shape > 50 or scale < 1e-6:
                    raise ValueError(f"Degenerate: shape={shape:.1f} scale={scale:.6f}")
                self.weibull_params[k] = (shape, loc, scale)
            except Exception as e:
                # Fallback: use 95th percentile gate as a logistic score
                p95 = float(np.percentile(dists, 95))
                self.weibull_params[k] = ("fallback_p95", p95, dists.std())
                print(f"    EVM class={k}: Weibull fallback ({e})")
        self.fitted = True
        print(f"  ✓ EVM  ({time.time()-t0:.2f}s)  tail={self.tail_size}  "
              f"classes={list(classes)}")
        return self

    def inclusion_score(self, X: np.ndarray) -> np.ndarray:
        """
        Returns per-sample EVM inclusion score in [0, 1].
        score=1 → confidently inside all known class regions.
        score=0 → outside every known class region (novel).
        """
        X = np.asarray(X, dtype=np.float64)
        per_class = []
        for k in sorted(self.weibull_params):
            mu   = self.class_means[k]
            dist = np.linalg.norm(X - mu, axis=1)
            params = self.weibull_params[k]
            if params[0] == "fallback_p95":
                _, p95, std = params
                # Sigmoid score centred on p95
                score = 1.0 / (1.0 + np.exp((dist - p95) / (std + 1e-6)))
            else:
                shape, loc, scale = params
                cdf   = weibull_min.cdf(dist, shape, loc=loc, scale=scale)
                score = 1.0 - np.clip(cdf, 0.0, 1.0)
            per_class.append(score)
        # Max inclusion across all classes (generous — novelty only if far from ALL classes)
        return np.stack(per_class, axis=1).max(1)

    def calibrate_threshold(self, X_val: np.ndarray, y_val: np.ndarray,
                             target_recall: float = 0.95) -> float:
        """
        Set EVM threshold so that target_recall% of known-class val samples
        have inclusion_score >= threshold.
        Returns the threshold value.
        """
        scores = self.inclusion_score(X_val)
        threshold = float(np.percentile(scores, (1 - target_recall) * 100))
        print(f"  EVM threshold calibrated: {threshold:.4f}  "
              f"(target recall={target_recall:.0%})")
        return threshold


# ── 5e. Evidential Deep Learning network ──────────────────────────────────────

class EvidentialNet(nn.Module):
    """
    Deep Evidential Classification. Outputs Dirichlet α (≥1).
    vacuity = K / α_0 in [0,1]. Used as a soft penalty, NOT a gate.
    """
    def __init__(self, in_features: int, n_classes: int,
                 hidden: Tuple = EDL_HIDDEN, dropout: float = EDL_DROPOUT):
        super().__init__()
        layers: List[nn.Module] = []; prev = in_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.ReLU(inplace=True), nn.Dropout(p=dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.softplus(self.net(x)) + 1.0

    def predict_with_uncertainty(self, X: np.ndarray):
        self.eval()
        with torch.no_grad():
            alpha = self(torch.from_numpy(X.astype(np.float32)).to(DEVICE)).cpu().numpy()
        alpha_0 = alpha.sum(1, keepdims=True)
        probs   = alpha / alpha_0
        K       = alpha.shape[1]
        vacuity = float(K) / alpha_0.ravel()
        epistemic = np.clip(
            sp_digamma(alpha_0.ravel() + 1)
            - (probs * sp_digamma(alpha + 1)).sum(1),
            0., None
        )
        aleatoric = np.clip(
            -np.sum(probs * np.log(probs + 1e-12), axis=1) - epistemic,
            0., None
        )
        return probs, vacuity, epistemic, aleatoric


def _edl_loss(alpha: torch.Tensor, y: torch.Tensor,
              epoch: int, n_classes: int) -> torch.Tensor:
    S    = alpha.sum(1, keepdim=True)
    y_oh = F.one_hot(y, n_classes).float()
    nll  = (y_oh * (torch.log(S) - torch.log(alpha))).sum(1).mean()
    anneal      = min(1.0, max(0.0, (epoch - EDL_ANNEAL_START) / 10.0))
    alpha_tilde = 1.0 + (alpha - 1.0) * (1.0 - y_oh)
    S_tilde     = alpha_tilde.sum(1, keepdim=True)
    ones        = torch.ones_like(alpha_tilde)
    kl = (
        torch.lgamma(S_tilde) - torch.lgamma(ones.sum(1, keepdim=True))
        - torch.lgamma(alpha_tilde).sum(1, keepdim=True)
        + ((alpha_tilde - 1) * (torch.digamma(alpha_tilde)
                                 - torch.digamma(S_tilde))).sum(1, keepdim=True)
    ).mean()
    return nll + EDL_KL_WEIGHT * anneal * kl


def train_evidential_net(X_tr: np.ndarray, y_tr: np.ndarray,
                          X_val: np.ndarray, y_val: np.ndarray,
                          n_classes: int):
    model = EvidentialNet(X_tr.shape[1], n_classes).to(DEVICE)
    opt   = optim.Adam(model.parameters(), lr=EDL_LR, weight_decay=EDL_WEIGHT_DECAY)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", patience=5,
                                                   factor=0.5, min_lr=1e-6)
    Xtr_t = torch.from_numpy(X_tr.astype(np.float32)).to(DEVICE)
    ytr_t = torch.from_numpy(y_tr.astype(np.int64)).to(DEVICE)
    Xva_t = torch.from_numpy(X_val.astype(np.float32)).to(DEVICE)
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t),
                        batch_size=EDL_BATCH_SIZE, shuffle=True)
    best_f1, best_w, es_ctr = -1., copy.deepcopy(model.state_dict()), 0
    train_losses, val_f1s = [], []
    for epoch in range(1, EDL_MAX_EPOCHS + 1):
        model.train(); ep_loss = 0.
        for Xb, yb in loader:
            opt.zero_grad()
            loss = _edl_loss(model(Xb), yb, epoch, n_classes)
            loss.backward(); opt.step()
            ep_loss += loss.item() * len(Xb)
        ep_loss /= len(Xtr_t); train_losses.append(ep_loss)
        model.eval()
        with torch.no_grad(): preds = model(Xva_t).cpu().numpy().argmax(1)
        vf1 = float(f1_score(y_val, preds, average="macro", zero_division=0))
        val_f1s.append(vf1); sched.step(vf1)
        if vf1 > best_f1 + 1e-5: best_f1, best_w, es_ctr = vf1, copy.deepcopy(model.state_dict()), 0
        else:
            es_ctr += 1
            if es_ctr >= EDL_ES_PATIENCE:
                print(f"    Early stop ep={epoch}  best F1={best_f1:.4f}"); break
        if epoch % 20 == 0 or epoch == 1:
            print(f"    Ep {epoch:>3}  loss={ep_loss:.4f}  val_F1={vf1:.4f}")
    model.load_state_dict(best_w); model.eval()
    print(f"  ✓ EDL best val F1={best_f1:.4f}")
    return model, train_losses, val_f1s


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6  ·  ANOMALY DETECTORS (unchanged — not the root cause)
# ─────────────────────────────────────────────────────────────────────────────

class VBGMMDetector:
    def fit(self, X: np.ndarray, y: np.ndarray) -> "VBGMMDetector":
        t0 = time.time(); self.bgmms: Dict[int, BayesianGaussianMixture] = {}
        for c in np.unique(y):
            Xc = X[y == c]; nc = min(VBGMM_MAX_COMPONENTS, max(1, len(Xc) // 10))
            bgm = BayesianGaussianMixture(
                n_components=nc, covariance_type="diag",
                weight_concentration_prior_type="dirichlet_process",
                weight_concentration_prior=1e-2,
                max_iter=VBGMM_MAX_ITER, random_state=RANDOM_SEED, reg_covar=1e-3
            )
            bgm.fit(Xc); self.bgmms[c] = bgm
            print(f"    VBGMM class={c}: {(bgm.weights_>1e-3).sum()}/{nc} active "
                  f"({time.time()-t0:.1f}s)")
        self.threshold = _pct(self.score(X), 99); return self

    def score(self, X: np.ndarray) -> np.ndarray:
        log_ps = np.stack([bgm.score_samples(X) for bgm in self.bgmms.values()], 1)
        return np.nan_to_num(-log_ps.max(1), nan=0., posinf=0., neginf=0.)

    def posterior_entropy(self, X: np.ndarray) -> np.ndarray:
        ents = [-(bgm.predict_proba(X) * np.log(bgm.predict_proba(X) + 1e-12)).sum(1)
                for bgm in self.bgmms.values()]
        return np.stack(ents, 1).min(1)


class MahalanobisDetector:
    def fit(self, X: np.ndarray, y: np.ndarray) -> "MahalanobisDetector":
        self.params: Dict[int, Tuple] = {}
        for c in np.unique(y):
            Xc  = X[y == c]; mu = Xc.mean(0)
            cov = np.cov(Xc, rowvar=False) + np.eye(Xc.shape[1]) * 1e-2
            try:    prec = np.linalg.inv(cov)
            except: prec = np.linalg.pinv(cov)
            self.params[c] = (mu, prec)
        self.threshold = _pct(self.score(X), 99); return self

    def score(self, X: np.ndarray) -> np.ndarray:
        dists = []
        for mu, prec in self.params.values():
            d = X - mu
            dists.append(np.sqrt(np.maximum(np.einsum("ni,ij,nj->n", d, prec, d), 0.)))
        return np.nan_to_num(np.stack(dists, 1).min(1), nan=0., posinf=0., neginf=0.)


class IsoForestDetector:
    def fit(self, X: np.ndarray, y: Optional[np.ndarray] = None) -> "IsoForestDetector":
        self.model = IsolationForest(n_estimators=200, contamination=0.02,
                                      n_jobs=-1, random_state=RANDOM_SEED).fit(X)
        self.threshold = _pct(self.score(X), 99); return self

    def score(self, X: np.ndarray) -> np.ndarray:
        return np.nan_to_num(-self.model.score_samples(X), nan=0., posinf=0., neginf=0.)


class ThreatScorer:
    """
    Empirical-Bayes weighted fusion of three anomaly detectors.
    Returns a normalised score in [0, 1].  Threshold set at 97th pct of training.
    """
    def __init__(self, det_v: VBGMMDetector, det_m: MahalanobisDetector,
                 det_i: IsoForestDetector, X_train: np.ndarray):
        self._dets = [("vbgmm", det_v), ("mahal", det_m), ("isoforest", det_i)]
        raw: Dict[str, np.ndarray] = {}
        for name, det in self._dets:
            s = det.score(X_train); lo, hi = _pct(s, 1), _pct(s, 99)
            if hi <= lo: hi = lo + 1.
            raw[name] = np.clip((s - lo) / (hi - lo + 1e-12), 0., 1.)
        variances = {n: float(v.var()) for n, v in raw.items()}
        total_v   = sum(variances.values()) + 1e-12
        self._weights  = {n: v / total_v for n, v in variances.items()}
        self._norm_lo  = {n: _pct(det.score(X_train), 1)  for n, det in self._dets}
        self._norm_hi  = {n: _pct(det.score(X_train), 99) for n, det in self._dets}
        for n in self._norm_lo:
            if self._norm_hi[n] <= self._norm_lo[n]: self._norm_hi[n] = self._norm_lo[n] + 1.
        raw_thr = _pct(self.compute(X_train), 97)
        self.threshold = max(float(raw_thr), THREAT_SCORE_THRESH)
        print(f"\n  Anomaly detector weights: "
              + "  ".join(f"{n}={w:.3f}" for n, w in self._weights.items()))
        print(f"  Threat threshold: {self.threshold:.4f}")
        self.vbgmm = det_v; self.iso = det_i
        self._iso_lo = self._norm_lo["isoforest"]
        self._iso_hi = self._norm_hi["isoforest"]

    def compute(self, X_sc: np.ndarray) -> np.ndarray:
        n = X_sc.shape[0] if X_sc.ndim > 1 else 1
        result = np.zeros(n, dtype=np.float64)
        for name, det in self._dets:
            lo, hi, w = self._norm_lo[name], self._norm_hi[name], self._weights[name]
            result += w * np.clip((det.score(X_sc) - lo) / (hi - lo + 1e-12), 0., 1.)
        return result


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7  ·  MODEL TRAINING
# ─────────────────────────────────────────────────────────────────────────────

def build_and_evaluate(X_sel: np.ndarray, y: np.ndarray,
                       classes_present: List[str]):
    print(f"\n{'='*60}\nMODEL TRAINING\n{'='*60}")
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_sel, y, test_size=0.20, stratify=y, random_state=RANDOM_SEED
    )
    _, X_val, _, y_val = train_test_split(
        X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=RANDOM_SEED
    )
    _, cnts = np.unique(y_tr, return_counts=True)
    k_smote = max(1, min(5, int(cnts.min()) - 1))
    X_sm, y_sm = SMOTE(random_state=RANDOM_SEED, k_neighbors=k_smote).fit_resample(X_tr, y_tr)
    print(f"  SMOTE k={k_smote}: train={X_sm.shape[0]:,}  val={X_val.shape[0]:,}  test={X_te.shape[0]:,}")
    n_cls = len(classes_present)

    rf = RandomForestClassifier(500, class_weight="balanced", max_features="sqrt",
                                 min_samples_leaf=2, random_state=RANDOM_SEED,
                                 n_jobs=-1, oob_score=True)
    rf.fit(X_sm, y_sm)
    yp_rf  = rf.predict(X_te)
    acc_rf = accuracy_score(y_te, yp_rf)
    f1_rf  = f1_score(y_te, yp_rf, average="macro", zero_division=0)
    print(f"\n  [A] RF   acc={acc_rf:.4f}  F1={f1_rf:.4f}  OOB={rf.oob_score_:.4f}")

    gbt = GradientBoostingClassifier(n_estimators=200, learning_rate=0.08,
                                      max_depth=5, subsample=0.8,
                                      min_samples_leaf=5, random_state=RANDOM_SEED)
    t0 = time.time(); gbt.fit(X_sm, y_sm)
    yp_gbt  = gbt.predict(X_te)
    acc_gbt = accuracy_score(y_te, yp_gbt)
    f1_gbt  = f1_score(y_te, yp_gbt, average="macro", zero_division=0)
    print(f"  [B] GBT  acc={acc_gbt:.4f}  F1={f1_gbt:.4f}  ({time.time()-t0:.1f}s)")

    lr_clf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000,
                                 random_state=RANDOM_SEED, n_jobs=-1)
    lr_clf.fit(X_sm, y_sm)
    yp_lr  = lr_clf.predict(X_te)
    acc_lr = accuracy_score(y_te, yp_lr)
    f1_lr  = f1_score(y_te, yp_lr, average="macro", zero_division=0)
    print(f"  [C] LR   acc={acc_lr:.4f}  F1={f1_lr:.4f}")

    print(f"\n  [D] EvidentialNet:")
    edl_model, dl_loss, dl_f1s = train_evidential_net(X_sm, y_sm, X_val, y_val, n_cls)
    yp_edl = edl_model.predict_with_uncertainty(X_te)[0].argmax(1)
    acc_edl = accuracy_score(y_te, yp_edl)
    f1_edl  = f1_score(y_te, yp_edl, average="macro", zero_division=0)
    print(f"  [D] EDL  acc={acc_edl:.4f}  F1={f1_edl:.4f}")

    # Temperature scaling for RF logits (calibration)
    rf_logits_val = rf.predict_proba(X_val)
    rf_logits_val = np.log(rf_logits_val.clip(1e-9, 1)) # log-probs as logits
    ts_calibrator = TemperatureScaler()
    try:
        ts_calibrator.fit(rf_logits_val, y_val)
    except Exception as e:
        print(f"  [WARN] Temperature scaling failed: {e}  (using T=1.0)")

    print(f"\n  Classification report (RF):")
    print(classification_report(y_te, yp_rf, target_names=classes_present, zero_division=0))

    return (rf, gbt, lr_clf, edl_model, ts_calibrator,
            X_te, y_te, X_val, y_val, X_sm, y_sm,
            yp_rf, acc_rf, f1_rf, acc_gbt, f1_gbt, acc_lr, f1_lr, acc_edl, f1_edl,
            dl_loss, dl_f1s)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8  ·  SOFT FUSION ENGINE  (THE CORE FIX)
# ─────────────────────────────────────────────────────────────────────────────

class SoftFusionEngine:
    """
    REPLACES all hard-gate stacking from v8.

    ARCHITECTURE:
      soft_score = w_clf × clf_confidence
                 + w_evm × evm_inclusion_score
                 + w_norm × (1 - normalised_anomaly_score)

    DECISION:
      soft_score <  open_set_threshold  →  OPEN_SET_UNKNOWN
      soft_score >= friendly_threshold  →  fast-path (FRIENDLY / BACKGROUND)
      otherwise                         →  temporal tracker path

    All thresholds calibrated on validation set.
    No EDL vacuity or EVM as a binary GATE — they are SCORES only.

    WHY THIS IS CORRECT:
      Multiple binary filters compound exponentially:
        P(pass N gates) = Π P(pass_i) → e.g. 0.85^4 = 0.52 for known-class signals
      A single soft score avoids this.  The calibrated threshold ensures that
      (1-recall_target)% of known-class val samples exceed the threshold.
    """
    def __init__(self, rf_clf, gbt_clf, gbp: GaussianBayesPosterior,
                 edl_model: EvidentialNet, evm: ExtremValueMachine,
                 threat_scorer: ThreatScorer, laplace: LaplaceApproximation,
                 ts_calibrator: TemperatureScaler,
                 classes_present: List[str],
                 open_set_threshold: float,
                 friendly_threshold: float):
        self.rf       = rf_clf
        self.gbt      = gbt_clf
        self.gbp      = gbp
        self.edl      = edl_model
        self.evm      = evm
        self.ts_det   = threat_scorer
        self.laplace  = laplace
        self.ts_cal   = ts_calibrator
        self.classes  = classes_present
        self.n        = len(classes_present)
        self.open_set_threshold = open_set_threshold
        self.friendly_threshold = friendly_threshold

    def score(self, X_sc: np.ndarray) -> Dict[str, Any]:
        """
        Full trace for one sample. Returns dict with all intermediate values.
        This is the canonical per-sample trace used for debugging.
        """
        eps = 1e-12

        # ── Classifier confidences ─────────────────────────────────────────
        rf_p  = self.rf.predict_proba(X_sc)[0].astype(np.float64) + eps
        gbt_p = self.gbt.predict_proba(X_sc)[0].astype(np.float64) + eps
        gbp_p = self.gbp.predict_proba(X_sc)[0].astype(np.float64) + eps

        # Geometric mean fusion (consensus — all must agree)
        combined   = (rf_p * gbt_p * gbp_p) ** (1.0 / 3.0)
        combined  /= combined.sum()
        winner_idx = int(combined.argmax())
        sorted_c   = np.sort(combined)[::-1]
        margin     = float(sorted_c[0] - sorted_c[1]) if self.n > 1 else 1.0

        # Temperature-calibrated confidence
        rf_logits  = np.log(rf_p.clip(1e-9, 1))
        cal_rf_p   = self.ts_cal.calibrate(rf_logits.reshape(1, -1))[0]
        clf_conf   = float(cal_rf_p.max() * (0.5 + 0.5 * margin))

        # ── EVM soft score ─────────────────────────────────────────────────
        evm_score = float(self.evm.inclusion_score(X_sc)[0])

        # ── Anomaly normality score (1 - normalised anomaly) ───────────────
        anomaly_raw  = float(self.ts_det.compute(X_sc)[0])
        normality    = float(1.0 - np.clip(anomaly_raw, 0., 1.))

        # ── EDL vacuity (soft penalty) ─────────────────────────────────────
        _, vacuity, ep_unc, al_unc = self.edl.predict_with_uncertainty(X_sc)
        edl_vacuity = float(vacuity[0])

        # ── Epistemic uncertainty ──────────────────────────────────────────
        norm_H  = float(-np.dot(combined, np.log(combined + eps)) / (np.log(self.n) + eps))
        vbgmm_e = float(self.ts_det.vbgmm.posterior_entropy(X_sc)[0])
        ep_lap  = self.laplace.predictive_variance(X_sc)
        iso_raw = float(self.ts_det.iso.score(X_sc)[0])
        iso_norm = float(np.clip(
            (iso_raw - self.ts_det._iso_lo) / (self.ts_det._iso_hi - self.ts_det._iso_lo + 1e-12),
            0., 1.
        ))
        epistemic = float(np.clip(
            0.35 * vbgmm_e / (np.log(VBGMM_MAX_COMPONENTS) + 1e-12)
            + 0.35 * iso_norm + 0.20 * norm_H + 0.10 * min(ep_lap * 10., 1.),
            0., 1.
        ))
        aleatoric = float(np.clip(norm_H * normality, 0., 1.))

        # ── SOFT FUSION SCORE ──────────────────────────────────────────────
        #  EDL vacuity acts as a soft penalty (not a gate):
        #  vacuity_penalty = 1 - edl_vacuity (high vacuity → lower soft_score)
        vacuity_penalty = 1.0 - edl_vacuity
        raw_soft = (FUSION_W_CLF      * clf_conf
                  + FUSION_W_EVM      * evm_score
                  + FUSION_W_NORMALITY * normality)
        # Apply vacuity as a multiplicative blend (gentle penalty)
        soft_score = float(raw_soft * (0.7 + 0.3 * vacuity_penalty))

        return {
            # Raw model outputs
            "rf_probs":           rf_p.round(4).tolist(),
            "gbt_probs":          gbt_p.round(4).tolist(),
            "gbp_probs":          gbp_p.round(4).tolist(),
            "combined_probs":     combined.round(4).tolist(),
            "winner":             self.classes[winner_idx],
            "winner_idx":         winner_idx,
            # Confidence components
            "clf_conf":           round(clf_conf, 4),
            "evm_score":          round(evm_score, 4),
            "normality":          round(normality, 4),
            "anomaly_raw":        round(anomaly_raw, 4),
            "edl_vacuity":        round(edl_vacuity, 4),
            # Uncertainty
            "epistemic":          round(epistemic, 4),
            "aleatoric":          round(aleatoric, 4),
            "predictive_entropy": round(norm_H, 4),
            # Final score
            "soft_score":         round(soft_score, 4),
            "open_set_threshold": round(self.open_set_threshold, 4),
            "friendly_threshold": round(self.friendly_threshold, 4),
            "margin":             round(margin, 4),
            # Threat
            "threat_score":       round(anomaly_raw, 4),
            "is_novel":           bool(soft_score < self.open_set_threshold),
            # Calibrated confidence (kept for backward compat)
            "calibrated_confidence": round(clf_conf, 4),
        }

    @classmethod
    def calibrate_thresholds(cls, engine_instance: "SoftFusionEngine",
                              X_val: np.ndarray, y_val: np.ndarray,
                              scaler_sel, selected_idx: np.ndarray,
                              target_recall: float = 0.95) -> Tuple[float, float]:
        """
        Compute open_set_threshold and friendly_threshold from validation data.
        open_set_threshold: (1-target_recall) quantile of soft_score on val known-class data.
        friendly_threshold: 50th percentile soft_score of high-confidence correct val predictions.
        """
        print(f"\n  Calibrating thresholds on validation set "
              f"(target_recall={target_recall:.0%}) ...")
        scores = []
        for i in range(len(X_val)):
            fv_full = np.zeros(N_FEATURES, dtype=np.float32)
            raw = scaler_sel.inverse_transform(X_val[i].reshape(1, -1))[0]
            for sp, oc in enumerate(selected_idx): fv_full[oc] = float(raw[sp])
            fv_sel = fv_full[selected_idx]
            X_sc   = np.nan_to_num(scaler_sel.transform(fv_sel.reshape(1, -1)),
                                    nan=0., posinf=0., neginf=0.)
            sc = engine_instance.score(X_sc)
            scores.append(sc["soft_score"])

        scores_arr = np.array(scores)
        open_thr   = float(np.percentile(scores_arr, (1 - target_recall) * 100))
        # Friendly threshold: median of known-class val scores
        friendly_thr = float(np.percentile(scores_arr, 40))
        print(f"    open_set_threshold  = {open_thr:.4f}  "
              f"(p{int((1-target_recall)*100)} of val scores)")
        print(f"    friendly_threshold  = {friendly_thr:.4f}  (p40 of val scores)")
        print(f"    score range: [{scores_arr.min():.4f}, {scores_arr.max():.4f}]"
              f"  mean={scores_arr.mean():.4f}")
        return open_thr, friendly_thr


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9  ·  EMITTER FINGERPRINTING & TEMPORAL TRACKER
# ─────────────────────────────────────────────────────────────────────────────

_HASH_STATE: List[Optional[np.ndarray]] = [None]


def emitter_hash(fv: np.ndarray) -> str:
    idx  = _HASH_STATE[0]
    fv_h = fv[idx] if idx is not None else fv
    qfp  = np.round(np.clip(fv_h, -HASH_CLIP, HASH_CLIP) * HASH_N_BINS).astype(np.int32)
    return hashlib.md5(qfp.tobytes()).hexdigest()[:16]


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    a = a.ravel().astype(np.float64);  b = b.ravel().astype(np.float64)
    return float(np.dot(a, b) / ((np.dot(a, a) * np.dot(b, b)) ** 0.5 + 1e-12))


@dataclass
class EmitterRecord:
    emitter_id:      str
    feature_history: deque    = field(default_factory=lambda: deque(maxlen=50))
    first_seen:      float    = field(default_factory=time.time)
    last_seen:       float    = field(default_factory=time.time)
    seen_count:      int      = 0
    threat_scores:   List[float] = field(default_factory=list)
    trust_score:     float    = 0.0
    promoted:        bool     = False
    auto_class:      Optional[str] = None
    auto_conf:       float    = 0.0

    def update(self, fv: np.ndarray, ts: float):
        self.feature_history.append(fv.copy())
        self.last_seen = time.time();  self.seen_count += 1
        self.threat_scores.append(float(ts))

    @property
    def mean_features(self) -> np.ndarray:
        return np.mean(np.stack(list(self.feature_history)), 0)

    @property
    def feature_variance(self) -> float:
        if len(self.feature_history) < 2: return 1.0
        stack = np.stack(list(self.feature_history))
        stds  = stack.std(0) + 1e-9
        return float(np.mean((stack / stds).var(0)))

    @property
    def mean_threat(self) -> float:
        return float(np.mean(self.threat_scores)) if self.threat_scores else 1.0

    def compute_trust(self) -> float:
        obs_t  = float(1.0 / (1.0 + np.exp(-(self.seen_count - TRUST_MIN_OBSERVATIONS) / 3.0)))
        stab_t = float(max(0., 1. - self.feature_variance / (TRUST_MAX_VARIANCE + 1e-9)))
        safe_t = float(max(0., 1. - self.mean_threat))
        vals   = [obs_t, stab_t, safe_t]
        self.trust_score = float(np.clip(len(vals) / sum(1./(v+1e-9) for v in vals), 0., 1.))
        return self.trust_score

    def is_trustworthy(self) -> bool:
        return (self.seen_count      >= TRUST_MIN_OBSERVATIONS and
                self.feature_variance <= TRUST_MAX_VARIANCE and
                self.mean_threat      <  HIGH_THREAT_THRESHOLD)


class TemporalTracker:
    def __init__(self): self.registry: Dict[str, EmitterRecord] = {}; self.total_obs = 0

    def observe(self, fv: np.ndarray, ts: float) -> EmitterRecord:
        eid = emitter_hash(fv)
        if eid not in self.registry: self.registry[eid] = EmitterRecord(emitter_id=eid)
        rec = self.registry[eid]; rec.update(fv, ts); rec.compute_trust()
        self.total_obs += 1; return rec

    def reset(self): self.registry = {}; self.total_obs = 0

    def summary(self) -> str:
        n   = len(self.registry)
        nt  = sum(1 for r in self.registry.values() if r.is_trustworthy())
        nth = sum(1 for r in self.registry.values() if r.mean_threat >= HIGH_THREAT_THRESHOLD)
        return f"Tracker: {n} emitters | trustworthy={nt} threat={nth} monitor={n-nt-nth}"


class FingerprintDatabase:
    def __init__(self, path: str):
        self.path = path
        self.trusted:    Dict[str, Dict] = {}
        self.suspicious: Dict[str, Dict] = {}
        self._load()

    def _load(self):
        if Path(self.path).exists():
            try:
                data = json.load(open(self.path))
                self.trusted    = data.get("trusted",    {})
                self.suspicious = data.get("suspicious", {})
                print(f"  DB loaded: {len(self.trusted)} trusted, "
                      f"{len(self.suspicious)} suspicious")
            except Exception: print("  DB corrupted — starting fresh")
        else: print("  DB: starting fresh")

    def save(self): json.dump({"trusted": self.trusted, "suspicious": self.suspicious},
                               open(self.path, "w"), indent=2)
    def reset(self): self.trusted = {}; self.suspicious = {}

    def match(self, fv: np.ndarray) -> Tuple[Optional[str], float, str]:
        best_sim, best_id, best_store = -1., None, ""
        for sname, db in (("trusted", self.trusted), ("suspicious", self.suspicious)):
            for eid, rec in db.items():
                sim = cosine_sim(fv, np.array(rec["fingerprint"]))
                if sim > best_sim: best_sim, best_id, best_store = sim, eid, sname
        return best_id, float(best_sim), best_store

    def add_trusted(self, eid: str, fv: np.ndarray,
                    seen: int, pred_class: str, conf: float):
        is_new = eid not in self.trusted
        label  = (f"AUTO_{pred_class.upper().replace(' ','_')}"
                  if conf >= AUTO_CLASSIFY_CONF and pred_class != BG_NAME
                  else (f"SAFE_UNKNOWN_{len(self.trusted)+1:03d}" if is_new
                        else self.trusted[eid]["label"]))
        self.trusted[eid] = {"fingerprint": fv.tolist(), "label": label,
                              "predicted_class": pred_class, "confidence": round(conf, 4),
                              "seen_count": seen,
                              "first_seen": self.trusted[eid]["first_seen"] if not is_new else time.time(),
                              "last_updated": time.time()}
        self.save()
        print(f"  {'✅ PROMOTED' if is_new else '🔄 UPDATED'} → {label}  "
              f"(class={pred_class}, conf={conf:.2f}, seen={seen})")

    def add_suspicious(self, eid: str, fv: np.ndarray, seen: int = 0):
        if eid not in self.suspicious:
            self.suspicious[eid] = {"fingerprint": fv.tolist(),
                                     "label": f"THREAT_{len(self.suspicious)+1:03d}",
                                     "seen_count": seen, "added_at": time.time()}
        else: self.suspicious[eid]["seen_count"] = seen
        self.save()

    def summary(self) -> str:
        return f"DB: {len(self.trusted)} trusted | {len(self.suspicious)} suspicious"

    def trusted_summary(self) -> str:
        if not self.trusted: return "  (empty)"
        return "\n".join(f"  {eid[:8]}.. → {r['label']:<35} "
                         f"conf={r.get('confidence',0):.2f}  seen={r.get('seen_count',0)}"
                         for eid, r in self.trusted.items())


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 10 ·  DECISION ENGINE  (CLEAN, UNIFIED PIPELINE)
# ─────────────────────────────────────────────────────────────────────────────

def make_classify_fn(fusion: SoftFusionEngine,
                     scaler_sel, selected_idx: np.ndarray,
                     fp_db: FingerprintDatabase,
                     temporal_tracker: TemporalTracker,
                     classes_present: List[str],
                     threat_scorer: ThreatScorer):
    """
    Factory returning classify_signal.

    Pipeline (soft decisions everywhere, single threshold for OPEN_SET):
      1. Scale features
      2. Compute soft_fusion_score via SoftFusionEngine.score()
      3. soft_score < open_set_threshold  →  OPEN_SET_UNKNOWN  (novel signal)
      4. soft_score >= friendly_threshold  →  FRIENDLY / BACKGROUND (fast path)
      5. Fingerprint DB match?             →  return cached label
      6. Threat path (anomaly score high)  →  POTENTIAL / CONFIRMED THREAT
      7. Trust promotion                   →  AUTO_* / SAFE_NEW_DRONE
      8. Default                           →  UNKNOWN_MONITOR
    """
    def classify_signal(fv_raw: np.ndarray,
                        return_bayes: bool = True) -> Dict[str, Any]:

        t_start = time.perf_counter()
        fv_raw  = np.nan_to_num(fv_raw.astype(np.float32), nan=0., posinf=0., neginf=0.)

        # ── 1. Scale ──────────────────────────────────────────────────────
        fv_sel = fv_raw[selected_idx] if len(fv_raw) == N_FEATURES else fv_raw
        X_sc   = np.nan_to_num(scaler_sel.transform(fv_sel.reshape(1, -1)),
                                nan=0., posinf=0., neginf=0.)

        # ── 2. Soft fusion score ──────────────────────────────────────────
        sc  = fusion.score(X_sc)
        ss  = sc["soft_score"]
        ts  = sc["threat_score"]

        result: Dict[str, Any] = {
            "label":       None,
            "bayesian":    sc if return_bayes else {},
            "emitter_id":  emitter_hash(fv_raw),
            "trust_score": 0.,
            "promoted":    False,
            "auto_class":  None,
            "soft_score":  round(ss, 4),
            "latency_ms":  0.,
        }

        # ── 3. Open-set check (SOFT — single threshold) ───────────────────
        if ss < fusion.open_set_threshold:
            result["label"] = "OPEN_SET_UNKNOWN"
            audit("open_set", soft_score=ss, threshold=fusion.open_set_threshold,
                  clf_conf=sc["clf_conf"], evm=sc["evm_score"])
            result["latency_ms"] = round((time.perf_counter() - t_start)*1000, 3)
            return result

        # ── 4. Confident known-class fast path ────────────────────────────
        winner = sc["winner"]
        if ss >= fusion.friendly_threshold:
            result["label"] = "BACKGROUND" if winner == BG_NAME else "FRIENDLY_DRONE"
            audit("fast_path", label=result["label"], soft_score=ss, winner=winner)
            result["latency_ms"] = round((time.perf_counter() - t_start)*1000, 3)
            return result

        # ── 5. Fingerprint DB match ───────────────────────────────────────
        match_id, sim, store = fp_db.match(fv_raw)
        if sim >= SIMILARITY_THRESHOLD and store == "trusted":
            db_lbl = fp_db.trusted[match_id].get("label", "TRUSTED_NEW_DRONE")
            result["label"] = db_lbl if db_lbl.startswith("AUTO_") else "TRUSTED_NEW_DRONE"
            if return_bayes:
                sc["db_match"]      = db_lbl
                sc["db_similarity"] = round(sim, 4)
            audit("db_match", label=result["label"], sim=sim)
            result["latency_ms"] = round((time.perf_counter() - t_start)*1000, 3)
            return result

        # ── 6. Temporal tracker ───────────────────────────────────────────
        rec = temporal_tracker.observe(fv_raw, ts)
        result["trust_score"] = float(rec.trust_score)
        result["emitter_id"]  = rec.emitter_id

        if ts >= threat_scorer.threshold or rec.mean_threat >= HIGH_THREAT_THRESHOLD:
            fp_db.add_suspicious(rec.emitter_id, rec.mean_features, rec.seen_count)
            result["label"] = ("CONFIRMED_THREAT" if rec.seen_count >= CONFIRMED_THREAT_OBS
                               else "POTENTIAL_THREAT")
            audit("threat", label=result["label"], ts=ts, seen=rec.seen_count)
            result["latency_ms"] = round((time.perf_counter() - t_start)*1000, 3)
            return result

        # ── 7. Trust promotion ────────────────────────────────────────────
        if rec.is_trustworthy() and not rec.promoted:
            mfeat = rec.mean_features
            msel  = mfeat[selected_idx] if len(mfeat) == N_FEATURES else mfeat
            X_m   = np.nan_to_num(scaler_sel.transform(msel.reshape(1, -1)),
                                   nan=0., posinf=0., neginf=0.)
            mean_sc = fusion.score(X_m)
            ac      = mean_sc["winner"]
            ac_conf = float(mean_sc["clf_conf"])
            fp_db.add_trusted(rec.emitter_id, rec.mean_features, rec.seen_count, ac, ac_conf)
            rec.promoted = True;  rec.auto_class = ac;  rec.auto_conf = ac_conf
            result["promoted"]   = True
            result["auto_class"] = ac
            result["label"]      = (f"AUTO_{ac.upper().replace(' ','_')}"
                                    if ac_conf >= AUTO_CLASSIFY_CONF and ac != BG_NAME
                                    else "SAFE_NEW_DRONE")
            audit("promoted", label=result["label"], auto_class=ac, conf=ac_conf)
            result["latency_ms"] = round((time.perf_counter() - t_start)*1000, 3)
            return result

        if rec.promoted or rec.emitter_id in fp_db.trusted:
            db_lbl = fp_db.trusted.get(rec.emitter_id, {}).get("label", "SAFE_NEW_DRONE")
            result["label"] = db_lbl if db_lbl.startswith("AUTO_") else "SAFE_NEW_DRONE"
        else:
            result["label"] = "UNKNOWN_MONITOR"

        audit("decision", label=result["label"], soft_score=ss, trust=rec.trust_score)
        result["latency_ms"] = round((time.perf_counter() - t_start)*1000, 3)
        return result

    return classify_signal


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 11 ·  DIAGNOSTIC TOOLS  (production-grade observability)
# ─────────────────────────────────────────────────────────────────────────────

def pipeline_trace(X_test: np.ndarray, y_test: np.ndarray,
                   fusion: SoftFusionEngine,
                   scaler_sel, selected_idx: np.ndarray,
                   classes_present: List[str],
                   n_samples: int = 10) -> pd.DataFrame:
    """
    Per-sample trace of every pipeline stage.
    Identifies which component is responsible for each decision.
    """
    print(f"\n{'='*70}\nPIPELINE TRACE  (first {n_samples} test samples)\n{'='*70}")
    print(f"  {'Idx':>3}  {'True':<16}  {'Winner':<16}  "
          f"{'clf_c':>6}  {'evm':>6}  {'norm':>6}  {'vac':>5}  "
          f"{'soft':>6}  {'Decision'}")
    print(f"  {'-'*100}")
    rows = []
    for i in range(min(n_samples, len(X_test))):
        fv_full = np.zeros(N_FEATURES, dtype=np.float32)
        raw = scaler_sel.inverse_transform(X_test[i].reshape(1, -1))[0]
        for sp, oc in enumerate(selected_idx): fv_full[oc] = float(raw[sp])
        fv_sel = fv_full[selected_idx]
        X_sc   = np.nan_to_num(scaler_sel.transform(fv_sel.reshape(1, -1)),
                                nan=0., posinf=0., neginf=0.)
        sc = fusion.score(X_sc)
        true_lbl = classes_present[y_test[i]]
        decision = ("OPEN_SET" if sc["soft_score"] < fusion.open_set_threshold
                    else ("FAST_PATH" if sc["soft_score"] >= fusion.friendly_threshold
                          else "TRACKER"))
        match = "✓" if sc["winner"] == true_lbl else "✗"
        print(f"  {i:>3}  {true_lbl:<16}  {sc['winner']:<16}  "
              f"  {sc['clf_conf']:>5.3f}  {sc['evm_score']:>5.3f}  "
              f"{sc['normality']:>5.3f}  {sc['edl_vacuity']:>4.2f}  "
              f"{sc['soft_score']:>5.3f}  {match} {decision}")
        rows.append({"idx": i, "true": true_lbl, "winner": sc["winner"],
                     "clf_conf": sc["clf_conf"], "evm_score": sc["evm_score"],
                     "normality": sc["normality"], "edl_vacuity": sc["edl_vacuity"],
                     "soft_score": sc["soft_score"], "decision": decision,
                     "correct": sc["winner"] == true_lbl})
    return pd.DataFrame(rows)


def rejection_analysis(X_test: np.ndarray, y_test: np.ndarray,
                       fusion: SoftFusionEngine,
                       scaler_sel, selected_idx: np.ndarray) -> Dict[str, float]:
    """
    Reports exactly which component is responsible for each rejection,
    as percentages.  Critical for identifying over-rejection.
    """
    print(f"\n{'='*60}\nREJECTION ANALYSIS\n{'='*60}")
    n_open_set = n_fast = n_tracker = 0
    clf_confs, evm_scores, normalities, soft_scores, vacuities = [], [], [], [], []
    for i in range(len(X_test)):
        fv_full = np.zeros(N_FEATURES, dtype=np.float32)
        raw = scaler_sel.inverse_transform(X_test[i].reshape(1, -1))[0]
        for sp, oc in enumerate(selected_idx): fv_full[oc] = float(raw[sp])
        fv_sel = fv_full[selected_idx]
        X_sc   = np.nan_to_num(scaler_sel.transform(fv_sel.reshape(1, -1)),
                                nan=0., posinf=0., neginf=0.)
        sc = fusion.score(X_sc)
        clf_confs.append(sc["clf_conf"]); evm_scores.append(sc["evm_score"])
        normalities.append(sc["normality"]); soft_scores.append(sc["soft_score"])
        vacuities.append(sc["edl_vacuity"])
        if sc["soft_score"] < fusion.open_set_threshold: n_open_set += 1
        elif sc["soft_score"] >= fusion.friendly_threshold: n_fast += 1
        else: n_tracker += 1

    N = len(X_test)
    stats = {
        "pct_open_set":          round(n_open_set / N * 100, 1),
        "pct_fast_path":         round(n_fast      / N * 100, 1),
        "pct_tracker":           round(n_tracker   / N * 100, 1),
        "mean_clf_conf":         round(float(np.mean(clf_confs)), 4),
        "mean_evm_score":        round(float(np.mean(evm_scores)), 4),
        "mean_normality":        round(float(np.mean(normalities)), 4),
        "mean_soft_score":       round(float(np.mean(soft_scores)), 4),
        "mean_edl_vacuity":      round(float(np.mean(vacuities)), 4),
        "open_set_threshold":    round(fusion.open_set_threshold, 4),
        "friendly_threshold":    round(fusion.friendly_threshold, 4),
    }
    print(f"  {'Component':<30} {'Value':>10}")
    print(f"  {'-'*40}")
    for k, v in stats.items():
        flag = ""
        if k == "pct_open_set" and v > 30:     flag = "  ⚠️  HIGH"
        if k == "mean_evm_score" and v < 0.2:  flag = "  ⚠️  EVM TOO LOW"
        if k == "mean_clf_conf" and v < 0.3:   flag = "  ⚠️  CONF TOO LOW"
        print(f"  {k:<30} {v:>10}  {flag}")
    return stats


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 12 ·  SIMULATION & EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

def _compute_synthetic_profiles(df: pd.DataFrame) -> Dict[str, Dict]:
    med = df.groupby("label_int")[ALL_FEATURE_NAMES].median()
    std = df.groupby("label_int")[ALL_FEATURE_NAMES].std()
    g   = lambda c, f, fb: float(med.loc[c, f]) if c in med.index else fb
    gs  = lambda c, f, fb: float(std.loc[c, f]) if c in std.index else fb
    c1  = 1
    return {
        "DJI_Neo_Threat": {
            "signal_power_db":  g(c1,"signal_power_db",-18) + 2.5*gs(c1,"signal_power_db",4),
            "spectral_entropy": g(c1,"spectral_entropy",5.5) + 2.0*gs(c1,"spectral_entropy",0.5),
            "bandwidth_hz":     g(c1,"bandwidth_hz",2e6) + 2.5*gs(c1,"bandwidth_hz",0.4e6),
            "ifreq_std":        g(c1,"ifreq_std",0.8) + 1.5*gs(c1,"ifreq_std",0.15),
            "noise_scale_frac": 0.12, "is_threat": True, "seed": 3001,
            "note": "OcuSync3 / wideband erratic",
        },
        "Harmless_Surveyor": {
            "signal_power_db":  g(c1,"signal_power_db",-18) - 1.2*gs(c1,"signal_power_db",4),
            "spectral_entropy": g(c1,"spectral_entropy",5.5) - 1.0*gs(c1,"spectral_entropy",0.5),
            "bandwidth_hz":     g(c1,"bandwidth_hz",2e6) - 2.0*gs(c1,"bandwidth_hz",0.4e6),
            "noise_scale_frac": 0.06, "is_threat": False, "seed": 3003,
            "note": "Stable narrowband surveyor",
        },
        "Autel_EVO3_Threat": {
            "signal_power_db":  g(c1,"signal_power_db",-18) + 1.5*gs(c1,"signal_power_db",4),
            "spectral_entropy": g(c1,"spectral_entropy",5.5) + 2.5*gs(c1,"spectral_entropy",0.5),
            "noise_scale_frac": 0.15, "is_threat": True, "seed": 3002,
            "note": "Aggressive FHSS",
        },
        "Delivery_Bot": {
            "signal_power_db":  g(c1,"signal_power_db",-18) - 0.5*gs(c1,"signal_power_db",4),
            "spectral_entropy": g(c1,"spectral_entropy",5.5) + 0.8*gs(c1,"spectral_entropy",0.5),
            "noise_scale_frac": 0.08, "is_threat": False, "seed": 3004,
            "note": "Urban delivery drone",
        },
    }


def _gen_obs(prof: Dict, df: pd.DataFrame, n: int) -> List[np.ndarray]:
    rng      = np.random.default_rng(prof["seed"])
    base_cls = 1 if prof.get("is_threat", False) else 0
    base_df  = df[df["label_int"] == base_cls][ALL_FEATURE_NAMES]
    base     = base_df.median().values.astype(np.float64) if len(base_df) > 0 else np.zeros(N_FEATURES)
    for feat, val in prof.items():
        if feat in FEAT_IDX: base[FEAT_IDX[feat]] = val
    noise_std = df[ALL_FEATURE_NAMES].std().values.astype(np.float64) * prof.get("noise_scale_frac", 0.1)
    if prof.get("is_threat", False): noise_std *= 1.3
    return [(base + rng.standard_normal(N_FEATURES)*noise_std).astype(np.float32) for _ in range(n)]


def run_synthetic_simulation(classify_signal, df: pd.DataFrame, n_obs: int) -> Dict:
    profiles = _compute_synthetic_profiles(df)
    print(f"\n{'='*65}\nSYNTHETIC SIMULATION  ({n_obs} obs per drone)\n{'='*65}")
    sim_results: Dict[str, List] = {}
    for name, prof in profiles.items():
        print(f"\n── {name}  [{prof['note']}]")
        decisions = []
        for step, fv in enumerate(_gen_obs(prof, df, n_obs), 1):
            dec = classify_signal(fv, return_bayes=True); decisions.append(dec)
            b   = dec.get("bayesian", {}); promo = " ← PROMOTED" if dec.get("promoted") else ""
            lbl = dec.get("label") or "None"
            print(f"  t={step:>2}  {DECISION_ICONS.get(lbl,'❓')} {lbl:<28}"
                  f"  ss={dec.get('soft_score',0):.3f}"
                  f"  clf={b.get('clf_conf',0):.3f}"
                  f"  evm={b.get('evm_score',0):.3f}"
                  f"  ts={b.get('threat_score',0):.3f}"
                  f"  trust={dec.get('trust_score',0):.3f}{promo}")
        sim_results[name] = decisions
        final = decisions[-1]; lbl = final.get("label") or "None"
        print(f"  FINAL: {DECISION_ICONS.get(lbl,'?')} {lbl}"
              f"  soft_score={final.get('soft_score',0):.3f}")
    return sim_results


def run_full_evaluation(X_te: np.ndarray, y_te: np.ndarray,
                        scaler_sel, selected_idx: np.ndarray,
                        classify_signal, classes_present: List[str]) -> Tuple:
    print(f"\n{'='*65}\nFULL SYSTEM EVALUATION\n{'='*65}")
    X_te_raw = scaler_sel.inverse_transform(X_te)
    X_full   = np.zeros((len(X_te_raw), N_FEATURES), dtype=np.float32)
    for sp, oc in enumerate(selected_idx): X_full[:, oc] = X_te_raw[:, sp].astype(np.float32)

    test_decs = []
    for i in range(len(X_full)):
        dec = classify_signal(X_full[i], return_bayes=True)
        dec["true_class"] = classes_present[y_te[i]]
        test_decs.append(dec)

    test_df = pd.DataFrame(test_decs)
    for col in ["calibrated_confidence", "clf_conf", "evm_score", "normality",
                "epistemic", "aleatoric", "predictive_entropy", "threat_score",
                "soft_score", "winner", "edl_vacuity", "margin"]:
        test_df[col] = test_df["bayesian"].apply(
            lambda b: b.get(col) if isinstance(b, dict) else None
        )

    open_mask  = test_df["label"] == "OPEN_SET_UNKNOWN"
    known_mask = ~test_df["label"].isin([
        "POTENTIAL_THREAT", "CONFIRMED_THREAT", "UNKNOWN_MONITOR",
        "SAFE_NEW_DRONE", "TRUSTED_NEW_DRONE", "OPEN_SET_UNKNOWN"
    ])
    correct = (
        (test_df.loc[known_mask, "winner"] == test_df.loc[known_mask, "true_class"]).mean()
        if known_mask.sum() > 0 else 0.0
    )
    false_alarm = test_df["label"].isin(["POTENTIAL_THREAT", "CONFIRMED_THREAT"]).mean()
    bg_recall   = (
        test_df[test_df["true_class"] == BG_NAME]["label"].eq("BACKGROUND").mean()
        if (test_df["true_class"] == BG_NAME).any() else 0.0
    )
    correct_idx = test_df.loc[known_mask][
        test_df.loc[known_mask, "winner"] == test_df.loc[known_mask, "true_class"]
    ].index
    mean_conf    = test_df.loc[correct_idx, "clf_conf"].mean() if len(correct_idx) > 0 else 0.
    open_frac    = float(open_mask.mean())

    print(f"\n  ┌{'─'*50}┐")
    print(f"  │  {'METRIC':<30} {'VALUE':>10}  │  {'TARGET':>8}")
    print(f"  ├{'─'*50}┤")
    ok = lambda v, t, hi=True: "✅" if (v >= t if hi else v <= t) else "❌"
    print(f"  │  {'Test set size':<30} {len(test_df):>10,}  │")
    print(f"  │  {'Known-drone accuracy':<30} {correct:>9.1%}  │  {ok(correct,0.80)} ≥80%")
    print(f"  │  {'Mean conf (correct)':<30} {mean_conf:>10.4f}  │  {ok(mean_conf,0.70)} ≥0.70")
    print(f"  │  {'False alarm rate':<30} {false_alarm:>9.1%}  │  {ok(false_alarm,0.10,hi=False)} ≤10%")
    print(f"  │  {'Background recall':<30} {bg_recall:>9.1%}  │  {ok(bg_recall,0.80)} ≥80%")
    print(f"  │  {'Open-set unknown frac':<30} {open_frac:>9.1%}  │")
    print(f"  └{'─'*50}┘")

    print(f"\n  Label distribution:")
    for lbl, cnt in test_df["label"].value_counts().items():
        print(f"    {DECISION_ICONS.get(lbl,'?')} {lbl:<30} {cnt:>5}  ({cnt/len(test_df):.1%})")

    test_df.to_csv("system_test_decisions_v9.csv", index=False)
    return test_df, known_mask, correct, false_alarm, bg_recall, mean_conf, open_frac


def make_dashboard(X_sel, y_mapped, mi, rf_clf, X_te, y_te, test_df, known_mask,
                   classes_present, acc_rf, f1_rf, acc_gbt, f1_gbt, acc_edl,
                   sim_results, threat_threshold, dl_loss, dl_f1s, selected_idx,
                   mean_conf, correct, false_alarm, bg_recall, open_frac,
                   rejection_stats: Dict):

    C = {"friendly": "#10B981", "threat": "#EF4444", "background": "#6B7280",
         "safe_new": "#3B82F6", "monitor": "#F59E0B", "bayesian": "#8B5CF6",
         "deep": "#EC4899", "auto": "#06B6D4", "open_set": "#9333EA", "gbt": "#F97316"}
    n_imp = len(rf_clf.feature_importances_)
    feat_nm = [ALL_FEATURE_NAMES[selected_idx[i]] for i in range(n_imp)]

    fig = plt.figure(figsize=(28, 30))
    gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.55, wspace=0.40)

    # — Panel 1: Feature importances —
    ax1 = fig.add_subplot(gs[0, :2])
    top_n = min(20, n_imp); ti = np.argsort(rf_clf.feature_importances_)[::-1][:top_n]
    ti_nm = [feat_nm[i] for i in ti]; imps = rf_clf.feature_importances_[ti]
    colors = ["#E11D48" if nm in FLIGHT_FEATURE_NAMES else
              "#F59E0B" if nm in COMM_FEATURE_NAMES else
              "#8B5CF6" if "energy" in nm or "band" in nm else
              "#3B82F6" if any(k in nm for k in ("freq","bandwidth","entropy","centroid")) else
              "#10B981" for nm in ti_nm]
    ax1.barh(ti_nm[::-1], imps[::-1], color=colors[::-1], height=0.70)
    ax1.set_xlabel("Gini importance", fontsize=10)
    ax1.set_title("Feature Importances — Multi-Modal v9 (FIXED)", fontsize=11)
    ax1.legend(handles=[Patch(facecolor="#10B981", label="RF Amplitude/IQ"),
                         Patch(facecolor="#3B82F6", label="RF Spectral"),
                         Patch(facecolor="#8B5CF6", label="RF Band"),
                         Patch(facecolor="#E11D48", label="Flight"),
                         Patch(facecolor="#F59E0B", label="Comm")], fontsize=8)

    # — Panel 2: Soft score distribution —
    ax2 = fig.add_subplot(gs[0, 2])
    for lbl, col in [("FRIENDLY_DRONE", C["friendly"]), ("BACKGROUND", C["background"]),
                      ("OPEN_SET_UNKNOWN", C["open_set"]), ("UNKNOWN_MONITOR", C["monitor"])]:
        vals = test_df.loc[test_df["label"] == lbl, "soft_score"].dropna()
        if len(vals): ax2.hist(vals, bins=25, alpha=0.65, density=True, color=col,
                               label=f"{lbl} (n={len(vals)})")
    if "open_set_threshold" in rejection_stats:
        ax2.axvline(rejection_stats["open_set_threshold"], color="red", ls="--", lw=2,
                    label=f"Open-set thr={rejection_stats['open_set_threshold']:.3f}")
    if "friendly_threshold" in rejection_stats:
        ax2.axvline(rejection_stats["friendly_threshold"], color="green", ls="--", lw=1.5,
                    label=f"Friendly thr={rejection_stats['friendly_threshold']:.3f}")
    ax2.set_title("Soft Fusion Score Distribution", fontsize=11)
    ax2.set_xlabel("soft_score"); ax2.legend(fontsize=7)

    # — Panel 3: Uncertainty scatter —
    ax3 = fig.add_subplot(gs[1, 0])
    lmap_plot = {"FRIENDLY_DRONE": C["friendly"], "BACKGROUND": C["background"],
                  "POTENTIAL_THREAT": C["threat"], "CONFIRMED_THREAT": C["threat"],
                  "OPEN_SET_UNKNOWN": C["open_set"], "UNKNOWN_MONITOR": C["monitor"],
                  "SAFE_NEW_DRONE": C["safe_new"], "AUTO_AR_DRONE": C["auto"]}
    for lbl, col in lmap_plot.items():
        m = test_df["label"] == lbl
        if m.sum() > 0:
            ax3.scatter(test_df.loc[m, "aleatoric"], test_df.loc[m, "epistemic"],
                        c=col, alpha=0.35, s=10, label=f"{lbl}(n={m.sum()})")
    ax3.set_xlabel("Aleatoric"); ax3.set_ylabel("Epistemic")
    ax3.set_title("Uncertainty Decomposition", fontsize=10); ax3.legend(fontsize=6)

    # — Panel 4: Trust curve —
    ax4 = fig.add_subplot(gs[1, 1])
    if "Harmless_Surveyor" in sim_results:
        sims = sim_results["Harmless_Surveyor"]
        ax4.plot(range(1,len(sims)+1), [d["trust_score"] for d in sims],
                 color=C["safe_new"], lw=2.5, marker="o", ms=5, label="Trust")
        ax4.plot(range(1,len(sims)+1), [d["soft_score"] for d in sims],
                 color=C["bayesian"], lw=2, ls="--", label="Soft score")
        ax4.plot(range(1,len(sims)+1),
                 [d["bayesian"].get("clf_conf",0) for d in sims],
                 color=C["gbt"], lw=1.5, ls=":", label="clf_conf")
        ax4.axhline(AUTO_CLASSIFY_CONF, color="orange", ls=":", lw=1.5,
                    label=f"AutoConf={AUTO_CLASSIFY_CONF}")
        ax4.set_ylim(0, 1.05); ax4.set_title("Harmless_Surveyor Trust Build-up", fontsize=10)
        ax4.legend(fontsize=7)

    # — Panel 5: Threat curve —
    ax5 = fig.add_subplot(gs[1, 2])
    if "DJI_Neo_Threat" in sim_results:
        sims = sim_results["DJI_Neo_Threat"]
        ax5.plot(range(1,len(sims)+1), [d["bayesian"].get("threat_score",0) for d in sims],
                 color=C["threat"], lw=2.5, marker="s", ms=5, label="Threat score")
        ax5.plot(range(1,len(sims)+1), [d["soft_score"] for d in sims],
                 color=C["bayesian"], lw=2, ls="--", label="Soft score")
        ax5.axhline(threat_threshold, color="black", ls="--", lw=1.5,
                    label=f"Thr={threat_threshold:.3f}")
        ax5.set_ylim(0, 1.05); ax5.set_title("DJI_Neo_Threat Detection", fontsize=10)
        ax5.legend(fontsize=7)

    # — Panel 6: EDL training —
    ax6 = fig.add_subplot(gs[2, 0])
    if dl_loss and dl_f1s:
        ep = range(1, len(dl_loss)+1); ax6t = ax6.twinx()
        ax6.plot(ep, dl_loss, color=C["deep"], lw=2, label="EDL loss")
        ax6t.plot(ep, dl_f1s, color=C["bayesian"], lw=2, ls="--", label="Val F1")
        ax6.set_title("EvidentialNet Training", fontsize=10)
        l1, lb1 = ax6.get_legend_handles_labels(); l2, lb2 = ax6t.get_legend_handles_labels()
        ax6.legend(l1+l2, lb1+lb2, fontsize=7)

    # — Panel 7: Confusion matrix —
    ax7 = fig.add_subplot(gs[2, 1])
    known_s = test_df[known_mask & test_df["winner"].notna()].copy()
    if len(known_s) > 0:
        pres = sorted(set(known_s["true_class"]) | set(known_s["winner"]))
        cm   = confusion_matrix(known_s["true_class"], known_s["winner"], labels=pres)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=[p[:8] for p in pres], yticklabels=[p[:8] for p in pres],
                    ax=ax7, cbar=False, annot_kws={"size": 9})
        ax7.set_title("Confusion Matrix (RF×GBT×GBP)", fontsize=10); ax7.tick_params(labelsize=7)

    # — Panel 8: Rejection breakdown —
    ax8 = fig.add_subplot(gs[2, 2])
    bars = {"Open-set": rejection_stats.get("pct_open_set", 0),
            "Fast-path": rejection_stats.get("pct_fast_path", 0),
            "Tracker":   rejection_stats.get("pct_tracker", 0)}
    bar_cols = [C["open_set"], C["friendly"], C["monitor"]]
    ax8.bar(list(bars.keys()), list(bars.values()), color=bar_cols, alpha=0.85)
    ax8.set_ylabel("% of test samples"); ax8.set_title("Pipeline Routing Breakdown", fontsize=10)
    for i, (k, v) in enumerate(bars.items()):
        ax8.text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=9)

    # — Panel 9: Model comparison + KPIs —
    ax9 = fig.add_subplot(gs[3, :])
    model_data = {"RF":  (acc_rf,  f1_rf),
                  "GBT": (acc_gbt, f1_gbt),
                  "EDL": (acc_edl, 0.)}
    x = np.arange(len(model_data)); w = 0.35
    accs = [v[0] for v in model_data.values()]; f1s = [v[1] for v in model_data.values()]
    ax9.bar(x - w/2, accs, w, label="Accuracy", color=C["friendly"], alpha=0.85)
    ax9.bar(x + w/2, f1s,  w, label="F1 Macro", color=C["bayesian"], alpha=0.85)
    ax9.set_xticks(x); ax9.set_xticklabels(list(model_data.keys()), fontsize=11)
    ax9.set_ylim(0, 1.2); ax9.legend(fontsize=9)
    for i, (a, f) in enumerate(zip(accs, f1s)):
        ax9.text(i-w/2, a+0.01, f"{a:.3f}", ha="center", fontsize=8)
        if f > 0: ax9.text(i+w/2, f+0.01, f"{f:.3f}", ha="center", fontsize=8)
    kpi_txt = (f"System KPIs:  Accuracy={correct:.1%}  Conf={mean_conf:.3f}  "
               f"FA={false_alarm:.1%}  BG_recall={bg_recall:.1%}  "
               f"OpenSet={open_frac:.1%}")
    ax9.set_title(kpi_txt, fontsize=10)

    fig.suptitle(
        "Real-Time AI Anti-Drone Detection  v9  — PRODUCTION READY\n"
        "BUG-1 FIXED: EVM consistent L2 metric  |  "
        "BUG-2 FIXED: tail_size=0.30  |  "
        "BUG-3 FIXED: soft fusion replaces hard-gate stacking",
        fontsize=10, fontweight="600"
    )
    out = "antidrone_dashboard_v9.png"
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
    print(f"✓ Dashboard → {out}")
    return out


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 13 ·  MAIN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print(f"\n{'█'*70}")
    print("  REAL-TIME AI ANTI-DRONE DETECTION  v9  —  PRODUCTION READY")
    print("  3 critical bugs fixed  |  Soft fusion  |  Calibrated thresholds")
    print(f"{'█'*70}\n")

    # ── 1. Dataset ───────────────────────────────────────────────────────────
    df = build_or_load_dataset(DATA_DIR)
    X_use, y_mapped, lmap, CLASSES_PRESENT, N_CLS = prepare_data(df)

    # ── 2. Feature selection ─────────────────────────────────────────────────
    X_sel, selected_idx, scaler_sel, mi = validate_and_select_features(X_use, y_mapped)
    _HASH_STATE[0] = selected_idx[:HASH_TOP_FEATURES]
    print(f"\n  Emitter hash: top-{HASH_TOP_FEATURES} MI features  bins={HASH_N_BINS}")

    # ── 3. Train/split ───────────────────────────────────────────────────────
    (rf_clf, gbt_clf, lr_clf, edl_model, ts_calibrator,
     X_te, y_te, X_val, y_val, X_sm, y_sm,
     yp_rf, acc_rf, f1_rf, acc_gbt, f1_gbt, acc_lr, f1_lr, acc_edl, f1_edl,
     dl_loss, dl_f1s) = build_and_evaluate(X_sel, y_mapped, CLASSES_PRESENT)

    # ── 4. GBP ───────────────────────────────────────────────────────────────
    print(f"\n{'='*60}\nGAUSSIAN BAYES POSTERIOR\n{'='*60}")
    gbp = GaussianBayesPosterior(temperature=GBP_TEMPERATURE).fit(X_sm, y_sm)
    gbp_acc = accuracy_score(y_te, gbp.predict(X_te))
    gbp_max_conf = gbp.predict_proba(X_te).max(1).mean()
    print(f"  GBP  acc={gbp_acc:.4f}  mean_max_conf={gbp_max_conf:.4f}")

    # ── 5. Laplace ───────────────────────────────────────────────────────────
    print(f"\n{'='*60}\nLAPLACE (epistemic uncertainty)\n{'='*60}")
    laplace = LaplaceApproximation().fit(lr_clf, X_sm, y_sm, N_CLS)

    # ── 6. EVM (FIXED) ───────────────────────────────────────────────────────
    print(f"\n{'='*60}\nEXTREME VALUE MACHINE  (v9 FIXED)\n{'='*60}")
    evm = ExtremValueMachine(tail_size=EVM_TAIL_SIZE).fit(X_sm, y_sm)
    # Verify EVM is not producing all-zeros
    evm_test_scores = evm.inclusion_score(X_sm[:20])
    print(f"  EVM sanity check (first 20 train samples):")
    print(f"    min={evm_test_scores.min():.4f}  max={evm_test_scores.max():.4f}"
          f"  mean={evm_test_scores.mean():.4f}")
    if evm_test_scores.mean() < 0.01:
        print("  ⚠️  WARNING: EVM still near-zero on training data — using fallback percentile mode")

    # ── 7. Anomaly detectors + threat scorer ─────────────────────────────────
    print(f"\n{'='*60}\nANOMALY DETECTORS\n{'='*60}")
    det_v = VBGMMDetector().fit(X_sm, y_sm)
    det_m = MahalanobisDetector().fit(X_sm, y_sm)
    det_i = IsoForestDetector().fit(X_sm)
    threat_scorer = ThreatScorer(det_v, det_m, det_i, X_sm)

    # ── 8. Build SoftFusionEngine (initial thresholds) ───────────────────────
    fusion = SoftFusionEngine(
        rf_clf, gbt_clf, gbp, edl_model, evm, threat_scorer, laplace, ts_calibrator,
        CLASSES_PRESENT,
        open_set_threshold=OPEN_SET_THRESHOLD,
        friendly_threshold=FRIENDLY_THRESHOLD,
    )

    # ── 9. Calibrate thresholds on validation set ────────────────────────────
    print(f"\n{'='*60}\nTHRESHOLD CALIBRATION (validation set)\n{'='*60}")
    open_thr, friendly_thr = SoftFusionEngine.calibrate_thresholds(
        fusion, X_val, y_val, scaler_sel, selected_idx, target_recall=0.95
    )
    fusion.open_set_threshold = open_thr
    fusion.friendly_threshold = friendly_thr

    # ── 10. DB + Tracker ─────────────────────────────────────────────────────
    fp_db   = FingerprintDatabase(DB_PATH)
    tracker = TemporalTracker()
    classify_signal = make_classify_fn(
        fusion, scaler_sel, selected_idx, fp_db, tracker, CLASSES_PRESENT, threat_scorer
    )
    print(f"\n✓ {fp_db.summary()}")

    # ── 11. Pipeline trace (diagnostic) ──────────────────────────────────────
    trace_df = pipeline_trace(X_te, y_te, fusion, scaler_sel, selected_idx, CLASSES_PRESENT, n_samples=15)
    trace_df.to_csv("pipeline_trace_v9.csv", index=False)

    # ── 12. Rejection analysis ────────────────────────────────────────────────
    rej_stats = rejection_analysis(X_te, y_te, fusion, scaler_sel, selected_idx)

    # ── 13. Synthetic simulation ──────────────────────────────────────────────
    n_obs = TRUST_MIN_OBSERVATIONS + 5
    sim_results = run_synthetic_simulation(classify_signal, df, n_obs)
    print(f"\n{tracker.summary()}")
    print(f"\n{'='*60}\nTRUSTED DB AFTER SIMULATION\n{'='*60}")
    print(fp_db.trusted_summary())

    # Reset before real evaluation
    fp_db.reset(); tracker.reset()
    print("\n  ✓ Reset before evaluation")

    # ── 14. Full evaluation ───────────────────────────────────────────────────
    fp_db.reset(); tracker.reset()
    (test_df, known_mask, correct, false_alarm, bg_recall,
     mean_conf, open_frac) = run_full_evaluation(
        X_te, y_te, scaler_sel, selected_idx, classify_signal, CLASSES_PRESENT
    )

    # ── 15. Dashboard ─────────────────────────────────────────────────────────
    try:
        make_dashboard(
            X_sel, y_mapped, mi, rf_clf, X_te, y_te, test_df, known_mask,
            CLASSES_PRESENT, acc_rf, f1_rf, acc_gbt, f1_gbt, acc_edl,
            sim_results, threat_scorer.threshold, dl_loss, dl_f1s, selected_idx,
            mean_conf, correct, false_alarm, bg_recall, open_frac, rej_stats
        )
        try:
            from google.colab import files; files.download("antidrone_dashboard_v9.png")
        except Exception: pass
    except Exception as e:
        print(f"\nDashboard error: {e}"); import traceback; traceback.print_exc()

    # ── 16. Save ──────────────────────────────────────────────────────────────
    fp_db.save()
    pd.DataFrame({"epoch": range(1, len(dl_loss)+1),
                  "edl_loss": dl_loss, "val_f1": dl_f1s}).to_csv(
        "edl_training_v9.csv", index=False)

    # ── Summary ───────────────────────────────────────────────────────────────
    sep = "=" * 72
    print(f"\n{sep}")
    print("ANTI-DRONE SYSTEM v9  —  PRODUCTION READY  —  FINAL SUMMARY")
    print(f"{sep}")
    print(f"""
ROOT CAUSES FIXED:
  BUG-1  EVM distance mismatch  (cosine fit + L2 inference → 9.8× scale error)
         FIX: Consistent L2 throughout.  inclusion_score validated on train data.

  BUG-2  EVM degenerate Weibull  (tail_size=0.05 → 8 samples → shape≈160)
         FIX: tail_size=0.30 (30% of class samples, well-sampled tail)
         Fallback to sigmoid logistic score when Weibull fit degenerates.

  BUG-3  Hard-gate stacking  (4 binary gates → P(pass)≈0.50 for known signals)
         FIX: Single soft fusion score + one calibrated threshold.
         soft_score = 0.55*clf + 0.25*evm + 0.20*normality
         Threshold calibrated at p5 of validation soft_score distribution.

ARCHITECTURE:
  Features  : RF(52) + Flight(18) + Comm(12) = {N_FEATURES} total
  Classifiers: RF + GBT + GBP (geometric mean consensus)
  Calibration: Temperature scaling on RF logits (val set)
  Open-set  : EVM soft score (not gate) + EDL vacuity penalty
  Uncertainty: Laplace (epistemic) + VBGMM entropy + IsoForest
  Adaptation : Temporal tracker + fingerprint DB (no retraining)
  Audit trail: {LOG_PATH}

MODELS
  RF          : acc={acc_rf:.4f}  F1={f1_rf:.4f}  OOB={rf_clf.oob_score_:.4f}
  GBT         : acc={acc_gbt:.4f}  F1={f1_gbt:.4f}
  GBP (τ=0.5): acc={gbp_acc:.4f}  mean_max_conf={gbp_max_conf:.4f}
  EDL         : acc={acc_edl:.4f}  F1={f1_edl:.4f}

SYSTEM EVALUATION
  Known-drone accuracy    : {correct:.2%}     ← target ≥80%
  Mean conf (correct)     : {mean_conf:.4f}   ← target ≥0.70
  False alarm rate        : {false_alarm:.2%}       ← target ≤10%
  Background recall       : {bg_recall:.2%}
  Open-set unknown frac   : {open_frac:.2%}

PIPELINE ROUTING (test set)
  Open-set  : {rej_stats.get('pct_open_set',0):.1f}%   (novel signals)
  Fast-path : {rej_stats.get('pct_fast_path',0):.1f}%   (high-confidence known)
  Tracker   : {rej_stats.get('pct_tracker',0):.1f}%   (low-confidence, monitored)

TRUSTED DB
{fp_db.trusted_summary()}
""")
    print(sep)
    print("v9 production system ready.")


✓ Imports ready  |  device=cpu  |  Python 3.12.13
✓ Features: RF=52 + flight=18 + comm=12 = 82

██████████████████████████████████████████████████████████████████████
  REAL-TIME AI ANTI-DRONE DETECTION  v9  —  PRODUCTION READY
  3 critical bugs fixed  |  Soft fusion  |  Calibrated thresholds
██████████████████████████████████████████████████████████████████████


BUILDING DATASET FROM /content/drive/MyDrive/DroneRF/DroneRF
✓ Saved 4,500 rows → dronerf_features_v9.csv

  Classes: 3
    [0] Background RF  (1500 samples)
    [1] AR Drone  (1500 samples)
    [2] Phantom Drone  (1500 samples)

FEATURE SELECTION
  Zero-variance dropped: 32
    Top- 3 PCs: 1.000  [✓]
    Top- 5 PCs: 1.000  [✓]
    Top-10 PCs: 1.000  [✓]

  Top-15 features by MI:
     1. spectral_rolloff_85                  0.6226  ★★
     2. energy_band3                         0.6156  ★★
     3. spectral_spread                      0.6051  ★★
     4. spectral_centroid                    0.5906  ★★
     5. ifreq_mean        

DEBUG:antidrone.v9:{"ts": 1775904854.5098, "event": "open_set", "soft_score": 0.5206, "threshold": 0.62499, "clf_conf": 0.5788, "evm": 1.0}
DEBUG:antidrone.v9:{"ts": 1775904854.6751, "event": "open_set", "soft_score": 0.491, "threshold": 0.62499, "clf_conf": 0.5215, "evm": 1.0}


  t= 1  ❓ OPEN_SET_UNKNOWN              ss=0.521  clf=0.579  evm=1.000  ts=0.787  trust=0.000
  t= 2  ❓ OPEN_SET_UNKNOWN              ss=0.491  clf=0.521  evm=1.000  ts=0.722  trust=0.000


DEBUG:antidrone.v9:{"ts": 1775904854.83, "event": "open_set", "soft_score": 0.4607, "threshold": 0.62499, "clf_conf": 0.5143, "evm": 0.8494}
DEBUG:antidrone.v9:{"ts": 1775904855.0073, "event": "open_set", "soft_score": 0.4897, "threshold": 0.62499, "clf_conf": 0.5518, "evm": 0.8816}


  t= 3  ❓ OPEN_SET_UNKNOWN              ss=0.461  clf=0.514  evm=0.849  ts=0.720  trust=0.000
  t= 4  ❓ OPEN_SET_UNKNOWN              ss=0.490  clf=0.552  evm=0.882  ts=0.683  trust=0.000


...

DEBUG:antidrone.v9:{"ts": 1775904863.9718, "event": "decision", "label": "UNKNOWN_MONITOR", "soft_score": 0.6505, "trust": 2.999999962776635e-09}
DEBUG:antidrone.v9:{"ts": 1775904864.1943, "event": "fast_path", "label": "BACKGROUND", "soft_score": 0.8178, "winner": "Background RF"}
DEBUG:antidrone.v9:{"ts": 1775904864.4039, "event": "fast_path", "label": "BACKGROUND", "soft_score": 0.819, "winner": "Background RF"}
DEBUG:antidrone.v9:{"ts": 1775904864.5807, "event": "decision", "label": "UNKNOWN_MONITOR", "soft_score": 0.7167, "trust": 2.999999962876144e-09}
DEBUG:antidrone.v9:{"ts": 1775904864.8142, "event": "decision", "label": "UNKNOWN_MONITOR", "soft_score": 0.6442, "trust": 2.999999958172642e-09}
DEBUG:antidrone.v9:{"ts": 1775904865.0694, "event": "fast_path", "label": "BACKGROUND", "soft_score": 0.8241, "winner": "Background RF"}
DEBUG:antidrone.v9:{"ts": 1775904865.3112, "event": "decision", "label": "UNKNOWN_MONITOR", "soft_score": 0.7028, "trust": 2.999999962675692e-09}
DEBUG:


  ┌──────────────────────────────────────────────────┐
  │  METRIC                              VALUE  │    TARGET
  ├──────────────────────────────────────────────────┤
  │  Test set size                         900  │
  │  Known-drone accuracy               98.8%  │  ✅ ≥80%
  │  Mean conf (correct)                0.9821  │  ✅ ≥0.70
  │  False alarm rate                    1.7%  │  ✅ ≤10%
  │  Background recall                  90.7%  │  ✅ ≥80%
  │  Open-set unknown frac              23.1%  │
  └──────────────────────────────────────────────────┘

  Label distribution:
    🟡 UNKNOWN_MONITOR                  353  (39.2%)
    ⚪ BACKGROUND                       273  (30.3%)
    ❓ OPEN_SET_UNKNOWN                 208  (23.1%)
    🟢 FRIENDLY_DRONE                    51  (5.7%)
    🔴 POTENTIAL_THREAT                  15  (1.7%)
✓ Dashboard → antidrone_dashboard_v9.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


ANTI-DRONE SYSTEM v9  —  PRODUCTION READY  —  FINAL SUMMARY

ROOT CAUSES FIXED:
  BUG-1  EVM distance mismatch  (cosine fit + L2 inference → 9.8× scale error)
         FIX: Consistent L2 throughout.  inclusion_score validated on train data.

  BUG-2  EVM degenerate Weibull  (tail_size=0.05 → 8 samples → shape≈160)
         FIX: tail_size=0.30 (30% of class samples, well-sampled tail)
         Fallback to sigmoid logistic score when Weibull fit degenerates.

  BUG-3  Hard-gate stacking  (4 binary gates → P(pass)≈0.50 for known signals)
         FIX: Single soft fusion score + one calibrated threshold.
         soft_score = 0.55*clf + 0.25*evm + 0.20*normality
         Threshold calibrated at p5 of validation soft_score distribution.

ARCHITECTURE:
  Features  : RF(52) + Flight(18) + Comm(12) = 82 total
  Classifiers: RF + GBT + GBP (geometric mean consensus)
  Calibration: Temperature scaling on RF logits (val set)
  Open-set  : EVM soft score (not gate) + EDL vacuity penalty
  Uncerta